In [1]:
# # Set up for running selenium in Google Colab
# %%shell
# sudo apt -y update
# sudo apt install -y wget curl unzip
# wget http://archive.ubuntu.com/ubuntu/pool/main/libu/libu2f-host/libu2f-udev_1.1.4-1_all.deb
# dpkg -i libu2f-udev_1.1.4-1_all.deb
# wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# dpkg -i google-chrome-stable_current_amd64.deb
# CHROME_DRIVER_VERSION=`curl -sS chromedriver.storage.googleapis.com/LATEST_RELEASE`
# wget -N https://chromedriver.storage.googleapis.com/$CHROME_DRIVER_VERSION/chromedriver_linux64.zip -P /tmp/
# unzip -o /tmp/chromedriver_linux64.zip -d /tmp/
# chmod +x /tmp/chromedriver
# mv /tmp/chromedriver /usr/local/bin/chromedriver
# pip install selenium
# pip install chromedriver-autoinstaller-fix

In [2]:
import chromedriver_autoinstaller_fix
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import time
import os
import shutil
import pandas as pd
import requests
from enum import StrEnum
from bs4 import BeautifulSoup
from datetime import datetime

In [3]:
PROXY_HOSTS=['154.6.99.159','38.62.221.210','154.6.99.65',
             '38.62.223.107','38.62.222.126','38.62.223.126',
             '38.62.223.53','154.6.96.179','38.62.223.173',
             '38.62.222.114','154.6.97.13','154.6.98.73','154.6.98.40',
             '154.6.96.44','38.62.220.93','154.6.99.194','38.62.221.13',
             '154.6.96.60','154.6.99.162','154.6.99.93','154.6.98.126',
             '38.62.221.49','38.62.220.172','154.6.96.26','38.62.221.2']

PROXY_PORT = 3128

current_try = 0

# Function to initialize the WebDriver
def initialize_driver(download_folder = r"D:\learning\thesis"):
    global current_try
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--ignore-ssl-errors=yes')
    chrome_options.add_argument('--ignore-certificate-errors')
#     proxy_host = PROXY_HOSTS[current_try%10]
#     chrome_options.add_argument(f'--proxy-server=http://{proxy_host}:{PROXY_PORT}')
    chrome_options.add_experimental_option('excludeSwitches', ['enable-logging'])
    chrome_options.add_experimental_option("prefs", {
      "download.default_directory": download_folder
      })

    driver = webdriver.Chrome(options = chrome_options)
    current_try += 1

    try:
        driver.get("https://leetcode.com/")
        return driver
    except Exception as e:
        print(f"Exception: {e}")
        driver.quit()
        return initialize_driver()

In [4]:
def get_contest_data(num_of_contest, is_weekly, page_num, driver):
    
    if is_weekly :
        contest_url = f'https://leetcode.com/contest/weekly-contest-{num_of_contest}/ranking/{page_num}'
    else :
        contest_url = f'https://leetcode.com/contest/biweekly-contest-{num_of_contest}/ranking/{page_num}'
    
    driver.get(contest_url)
        
    time.sleep(2)
    
    try:
        table_div = WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
    except:
        time.sleep(30)
        driver.get(contest_url)
        time.sleep(2)
        table_div = WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
      
    
    for row in table_div.find_elements(By.TAG_NAME, 'tr')[1:]:
        get_row_data(row, contest_url, num_of_contest, is_weekly, driver)
        

In [5]:
def get_row_data(row_element, contest_url, num_of_contest, is_weekly, driver):
    column_elements = row_element.find_elements(By.TAG_NAME, 'td')
    
    rank = column_elements[0].get_attribute('innerHTML')
    country = column_elements[1].find_element(By.CLASS_NAME, 'country-name').get_attribute("data-original-title")
    username = column_elements[1].find_element(By.CLASS_NAME, 'ranking-username').get_attribute("title")
    score = column_elements[2].get_attribute('innerHTML')
    finish_time = get_datetime_from_string(column_elements[3].get_attribute('innerHTML'))
    
    if country :
    
        questions_data = []

        for code_column in column_elements[4:] :
            questions_data = questions_data + list(get_code_data(code_column, driver))

        leetcode_code_df.loc[len(leetcode_code_df.index)] = [username, country, contest_url, num_of_contest, finish_time,
                                                             is_weekly, rank, score, *questions_data]
        print("Finished scraping ", username, "in contest ", num_of_contest)

In [6]:
def get_datetime_from_string(minutes_str) :
    hours, minutes, seconds = map(int, minutes_str.split()[0].split(':'))
    return datetime(year=1, month=1, day=1, hour=hours, minute=minutes, second=seconds).time()

In [7]:
def get_code_data(code_column, driver):
    
    if not code_column.get_attribute('innerHTML') :
        return None, None, None
           
    if(code_column.find_elements(By.TAG_NAME, 'svg')) :
        code_language_element = code_column.find_element(By.TAG_NAME, 'svg')
    else :
        code_language_element = code_column.find_element(By.CLASS_NAME, 'fa-file-code-o')
    
    code_language = get_code_language_from_image(code_language_element.get_attribute("innerHTML"))
    
    code = None if code_language == 'unknown' else get_code_from_modal(code_language_element, driver)
    
    finish_time = get_datetime_from_string(code_column.text)
    
    return code_language, code, finish_time

In [8]:
class SVG_CODE(StrEnum):
    C = '<path fill="#659AD3" d="M115.4 30.7L67.1 2.9c-.8-.5-1.9-.7-3.1-.7-1.2 0-2.3.3-3.1.7l-48 27.9c-1.7 1-2.9 3.5-2.9 5.4v55.7c0 1.1.2 2.4 1 3.5l106.8-62c-.6-1.2-1.5-2.1-2.4-2.7z"></path><path fill="#03599C" d="M10.7 95.3c.5.8 1.2 1.5 1.9 1.9l48.2 27.9c.8.5 1.9.7 3.1.7 1.2 0 2.3-.3 3.1-.7l48-27.9c1.7-1 2.9-3.5 2.9-5.4V36.1c0-.9-.1-1.9-.6-2.8l-106.6 62z"></path><path fill="#fff" d="M85.3 76.1C81.1 83.5 73.1 88.5 64 88.5c-13.5 0-24.5-11-24.5-24.5s11-24.5 24.5-24.5c9.1 0 17.1 5 21.3 12.5l13-7.5c-6.8-11.9-19.6-20-34.3-20-21.8 0-39.5 17.7-39.5 39.5s17.7 39.5 39.5 39.5c14.6 0 27.4-8 34.2-19.8l-12.9-7.6z"></path>'
    C_PLUS = '<path fill="#D26383" d="M115.4 30.7L67.1 2.9c-.8-.5-1.9-.7-3.1-.7-1.2 0-2.3.3-3.1.7l-48 27.9c-1.7 1-2.9 3.5-2.9 5.4v55.7c0 1.1.2 2.4 1 3.5l106.8-62c-.6-1.2-1.5-2.1-2.4-2.7z"></path><path fill="#9C033A" d="M10.7 95.3c.5.8 1.2 1.5 1.9 1.9l48.2 27.9c.8.5 1.9.7 3.1.7 1.2 0 2.3-.3 3.1-.7l48-27.9c1.7-1 2.9-3.5 2.9-5.4V36.1c0-.9-.1-1.9-.6-2.8l-106.6 62z"></path><path fill="#ffffff" d="M85.3 76.1C81.1 83.5 73.1 88.5 64 88.5c-13.5 0-24.5-11-24.5-24.5s11-24.5 24.5-24.5c9.1 0 17.1 5 21.3 12.5l13-7.5c-6.8-11.9-19.6-20-34.3-20-21.8 0-39.5 17.7-39.5 39.5s17.7 39.5 39.5 39.5c14.6 0 27.4-8 34.2-19.8l-12.9-7.6z"></path><path d="M82.1 61.8h5.2v-5.3h4.4v5.3H97v4.4h-5.3v5.2h-4.4v-5.2h-5.2v-4.4zm18.5 0h5.2v-5.3h4.4v5.3h5.3v4.4h-5.3v5.2h-4.4v-5.2h-5.2v-4.4z" fill="#ffffff"></path>'
    C_SHARP = '<path fill="#9B4F96" d="M115.4 30.7L67.1 2.9c-.8-.5-1.9-.7-3.1-.7-1.2 0-2.3.3-3.1.7l-48 27.9c-1.7 1-2.9 3.5-2.9 5.4v55.7c0 1.1.2 2.4 1 3.5l106.8-62c-.6-1.2-1.5-2.1-2.4-2.7z"></path><path fill="#68217A" d="M10.7 95.3c.5.8 1.2 1.5 1.9 1.9l48.2 27.9c.8.5 1.9.7 3.1.7 1.2 0 2.3-.3 3.1-.7l48-27.9c1.7-1 2.9-3.5 2.9-5.4V36.1c0-.9-.1-1.9-.6-2.8l-106.6 62z"></path><path fill="#ffffff" d="M85.3 76.1C81.1 83.5 73.1 88.5 64 88.5c-13.5 0-24.5-11-24.5-24.5s11-24.5 24.5-24.5c9.1 0 17.1 5 21.3 12.5l13-7.5c-6.8-11.9-19.6-20-34.3-20-21.8 0-39.5 17.7-39.5 39.5s17.7 39.5 39.5 39.5c14.6 0 27.4-8 34.2-19.8l-12.9-7.6zM97 66.2l.9-4.3h-4.2v-4.7h5.1L100 51h4.9l-1.2 6.1h3.8l1.2-6.1h4.8l-1.2 6.1h2.4v4.7h-3.3l-.9 4.3h4.2v4.7h-5.1l-1.2 6h-4.9l1.2-6h-3.8l-1.2 6h-4.8l1.2-6h-2.4v-4.7H97zm4.8 0h3.8l.9-4.3h-3.8l-.9 4.3z"></path>'
    JAVA = '<path fill="#0074BD" d="M47.617 98.12s-4.767 2.774 3.397 3.71c9.892 1.13 14.947.968 25.845-1.092 0 0 2.871 1.795 6.873 3.351-24.439 10.47-55.308-.607-36.115-5.969zm-2.988-13.665s-5.348 3.959 2.823 4.805c10.567 1.091 18.91 1.18 33.354-1.6 0 0 1.993 2.025 5.132 3.131-29.542 8.64-62.446.68-41.309-6.336z"></path><path fill="#EA2D2E" d="M69.802 61.271c6.025 6.935-1.58 13.17-1.58 13.17s15.289-7.891 8.269-17.777c-6.559-9.215-11.587-13.792 15.635-29.58 0 .001-42.731 10.67-22.324 34.187z"></path><path fill="#0074BD" d="M102.123 108.229s3.529 2.91-3.888 5.159c-14.102 4.272-58.706 5.56-71.094.171-4.451-1.938 3.899-4.625 6.526-5.192 2.739-.593 4.303-.485 4.303-.485-4.953-3.487-32.013 6.85-13.743 9.815 49.821 8.076 90.817-3.637 77.896-9.468zM49.912 70.294s-22.686 5.389-8.033 7.348c6.188.828 18.518.638 30.011-.326 9.39-.789 18.813-2.474 18.813-2.474s-3.308 1.419-5.704 3.053c-23.042 6.061-67.544 3.238-54.731-2.958 10.832-5.239 19.644-4.643 19.644-4.643zm40.697 22.747c23.421-12.167 12.591-23.86 5.032-22.285-1.848.385-2.677.72-2.677.72s.688-1.079 2-1.543c14.953-5.255 26.451 15.503-4.823 23.725 0-.002.359-.327.468-.617z"></path><path fill="#EA2D2E" d="M76.491 1.587S89.459 14.563 64.188 34.51c-20.266 16.006-4.621 25.13-.007 35.559-11.831-10.673-20.509-20.07-14.688-28.815C58.041 28.42 81.722 22.195 76.491 1.587z"></path><path fill="#0074BD" d="M52.214 126.021c22.476 1.437 57-.8 57.817-11.436 0 0-1.571 4.032-18.577 7.231-19.186 3.612-42.854 3.191-56.887.874 0 .001 2.875 2.381 17.647 3.331z"></path>'
    PYTHON = '<linearGradient id="python-original-a" gradientUnits="userSpaceOnUse" x1="70.252" y1="1237.476" x2="170.659" y2="1151.089" gradientTransform="matrix(.563 0 0 -.568 -29.215 707.817)"><stop offset="0" stop-color="#5A9FD4"></stop><stop offset="1" stop-color="#306998"></stop></linearGradient><linearGradient id="python-original-b" gradientUnits="userSpaceOnUse" x1="209.474" y1="1098.811" x2="173.62" y2="1149.537" gradientTransform="matrix(.563 0 0 -.568 -29.215 707.817)"><stop offset="0" stop-color="#FFD43B"></stop><stop offset="1" stop-color="#FFE873"></stop></linearGradient><path fill="url(#python-original-a)" d="M63.391 1.988c-4.222.02-8.252.379-11.8 1.007-10.45 1.846-12.346 5.71-12.346 12.837v9.411h24.693v3.137H29.977c-7.176 0-13.46 4.313-15.426 12.521-2.268 9.405-2.368 15.275 0 25.096 1.755 7.311 5.947 12.519 13.124 12.519h8.491V67.234c0-8.151 7.051-15.34 15.426-15.34h24.665c6.866 0 12.346-5.654 12.346-12.548V15.833c0-6.693-5.646-11.72-12.346-12.837-4.244-.706-8.645-1.027-12.866-1.008zM50.037 9.557c2.55 0 4.634 2.117 4.634 4.721 0 2.593-2.083 4.69-4.634 4.69-2.56 0-4.633-2.097-4.633-4.69-.001-2.604 2.073-4.721 4.633-4.721z" transform="translate(0 10.26)"></path><path fill="url(#python-original-b)" d="M91.682 28.38v10.966c0 8.5-7.208 15.655-15.426 15.655H51.591c-6.756 0-12.346 5.783-12.346 12.549v23.515c0 6.691 5.818 10.628 12.346 12.547 7.816 2.297 15.312 2.713 24.665 0 6.216-1.801 12.346-5.423 12.346-12.547v-9.412H63.938v-3.138h37.012c7.176 0 9.852-5.005 12.348-12.519 2.578-7.735 2.467-15.174 0-25.096-1.774-7.145-5.161-12.521-12.348-12.521h-9.268zM77.809 87.927c2.561 0 4.634 2.097 4.634 4.692 0 2.602-2.074 4.719-4.634 4.719-2.55 0-4.633-2.117-4.633-4.719 0-2.595 2.083-4.692 4.633-4.692z" transform="translate(0 10.26)"></path><radialGradient id="python-original-c" cx="1825.678" cy="444.45" r="26.743" gradientTransform="matrix(0 -.24 -1.055 0 532.979 557.576)" gradientUnits="userSpaceOnUse"><stop offset="0" stop-color="#B8B8B8" stop-opacity=".498"></stop><stop offset="1" stop-color="#7F7F7F" stop-opacity="0"></stop></radialGradient><path opacity=".444" fill="url(#python-original-c)" d="M97.309 119.597c0 3.543-14.816 6.416-33.091 6.416-18.276 0-33.092-2.873-33.092-6.416 0-3.544 14.815-6.417 33.092-6.417 18.275 0 33.091 2.872 33.091 6.417z"></path>'
    JAVASCRIPT = '<path fill="#F0DB4F" d="M1.408 1.408h125.184v125.185H1.408z"></path><path fill="#323330" d="M116.347 96.736c-.917-5.711-4.641-10.508-15.672-14.981-3.832-1.761-8.104-3.022-9.377-5.926-.452-1.69-.512-2.642-.226-3.665.821-3.32 4.784-4.355 7.925-3.403 2.023.678 3.938 2.237 5.093 4.724 5.402-3.498 5.391-3.475 9.163-5.879-1.381-2.141-2.118-3.129-3.022-4.045-3.249-3.629-7.676-5.498-14.756-5.355l-3.688.477c-3.534.893-6.902 2.748-8.877 5.235-5.926 6.724-4.236 18.492 2.975 23.335 7.104 5.332 17.54 6.545 18.873 11.531 1.297 6.104-4.486 8.08-10.234 7.378-4.236-.881-6.592-3.034-9.139-6.949-4.688 2.713-4.688 2.713-9.508 5.485 1.143 2.499 2.344 3.63 4.26 5.795 9.068 9.198 31.76 8.746 35.83-5.176.165-.478 1.261-3.666.38-8.581zM69.462 58.943H57.753l-.048 30.272c0 6.438.333 12.34-.714 14.149-1.713 3.558-6.152 3.117-8.175 2.427-2.059-1.012-3.106-2.451-4.319-4.485-.333-.584-.583-1.036-.667-1.071l-9.52 5.83c1.583 3.249 3.915 6.069 6.902 7.901 4.462 2.678 10.459 3.499 16.731 2.059 4.082-1.189 7.604-3.652 9.448-7.401 2.666-4.915 2.094-10.864 2.07-17.444.06-10.735.001-21.468.001-32.237z"></path>'
    TYPESCRIPT = '<path fill="#fff" d="M22.67 47h99.67v73.67H22.67z"></path><path fill="#007acc" d="M1.5 63.91v62.5h125v-125H1.5zm100.73-5a15.56 15.56 0 017.82 4.5 20.58 20.58 0 013 4c0 .16-5.4 3.81-8.69 5.85-.12.08-.6-.44-1.13-1.23a7.09 7.09 0 00-5.87-3.53c-3.79-.26-6.23 1.73-6.21 5a4.58 4.58 0 00.54 2.34c.83 1.73 2.38 2.76 7.24 4.86 8.95 3.85 12.78 6.39 15.16 10 2.66 4 3.25 10.46 1.45 15.24-2 5.2-6.9 8.73-13.83 9.9a38.32 38.32 0 01-9.52-.1 23 23 0 01-12.72-6.63c-1.15-1.27-3.39-4.58-3.25-4.82a9.34 9.34 0 011.15-.73L82 101l3.59-2.08.75 1.11a16.78 16.78 0 004.74 4.54c4 2.1 9.46 1.81 12.16-.62a5.43 5.43 0 00.69-6.92c-1-1.39-3-2.56-8.59-5-6.45-2.78-9.23-4.5-11.77-7.24a16.48 16.48 0 01-3.43-6.25 25 25 0 01-.22-8c1.33-6.23 6-10.58 12.82-11.87a31.66 31.66 0 019.49.26zm-29.34 5.24v5.12H56.66v46.23H45.15V69.26H28.88v-5a49.19 49.19 0 01.12-5.17C29.08 59 39 59 51 59h21.83z"></path>'
    GO = '<g fill="#00acd7" fill-rule="evenodd"><path d="M11.156 54.829c-.243 0-.303-.122-.182-.303l1.273-1.637c.12-.182.424-.303.666-.303H34.55c.243 0 .303.182.182.364l-1.03 1.576c-.121.181-.424.363-.606.363zM2.004 60.404c-.242 0-.303-.12-.182-.303l1.273-1.636c.121-.182.424-.303.667-.303h27.636c.242 0 .364.182.303.364l-.485 1.454c-.06.243-.303.364-.545.364zM16.67 65.98c-.242 0-.302-.182-.181-.364l.848-1.515c.122-.182.364-.363.607-.363h12.12c.243 0 .364.181.364.424l-.12 1.454c0 .243-.243.425-.425.425zM79.58 53.738c-3.819.97-6.425 1.697-10.182 2.666-.91.243-.97.303-1.758-.606-.909-1.03-1.576-1.697-2.848-2.303-3.819-1.878-7.516-1.333-10.97.91-4.121 2.666-6.242 6.605-6.182 11.514.06 4.849 3.394 8.849 8.182 9.516 4.121.545 7.576-.91 10.303-4 .545-.667 1.03-1.394 1.636-2.243H56.064c-1.272 0-1.575-.788-1.151-1.818.788-1.879 2.242-5.03 3.09-6.606.183-.364.607-.97 1.516-.97h22.06c-.12 1.637-.12 3.273-.363 4.91-.667 4.363-2.303 8.363-4.97 11.878-4.364 5.758-10.06 9.333-17.273 10.303-5.939.788-11.454-.364-16.302-4-4.485-3.394-7.03-7.879-7.697-13.454-.788-6.606 1.151-12.546 5.151-17.758 4.303-5.636 10-9.212 16.97-10.485 5.697-1.03 11.151-.363 16.06 2.97 3.212 2.121 5.515 5.03 7.03 8.545.364.546.122.849-.606 1.03z"></path><path d="M99.64 87.253c-5.515-.122-10.546-1.697-14.788-5.334-3.576-3.09-5.818-7.03-6.545-11.697-1.091-6.848.787-12.909 4.909-18.302 4.424-5.819 9.757-8.849 16.97-10.122 6.181-1.09 12-.484 17.272 3.091 4.788 3.273 7.757 7.697 8.545 13.515 1.03 8.182-1.333 14.849-6.97 20.546-4 4.06-8.909 6.606-14.545 7.757-1.636.303-3.273.364-4.848.546zm14.424-24.485c-.06-.788-.06-1.394-.182-2-1.09-6-6.606-9.394-12.363-8.06-5.637 1.272-9.273 4.848-10.606 10.545-1.091 4.727 1.212 9.515 5.575 11.454 3.334 1.455 6.667 1.273 9.879-.363 4.788-2.485 7.394-6.364 7.697-11.576z" fill-rule="nonzero"></path></g>'
    SWIFT = '<path fill="#f05138" d="M126.33 34.06a39.32 39.32 0 00-.79-7.83 28.78 28.78 0 00-2.65-7.58 28.84 28.84 0 00-4.76-6.32 23.42 23.42 0 00-6.62-4.55 27.27 27.27 0 00-7.68-2.53c-2.65-.51-5.56-.51-8.21-.76H30.25a45.46 45.46 0 00-6.09.51 21.82 21.82 0 00-5.82 1.52c-.53.25-1.32.51-1.85.76a33.82 33.82 0 00-5 3.28c-.53.51-1.06.76-1.59 1.26a22.41 22.41 0 00-4.76 6.32 23.61 23.61 0 00-2.65 7.58 78.5 78.5 0 00-.79 7.83v60.39a39.32 39.32 0 00.79 7.83 28.78 28.78 0 002.65 7.58 28.84 28.84 0 004.76 6.32 23.42 23.42 0 006.62 4.55 27.27 27.27 0 007.68 2.53c2.65.51 5.56.51 8.21.76h63.22a45.08 45.08 0 008.21-.76 27.27 27.27 0 007.68-2.53 30.13 30.13 0 006.62-4.55 22.41 22.41 0 004.76-6.32 23.61 23.61 0 002.65-7.58 78.49 78.49 0 00.79-7.83V34.06z"></path><path fill="#fefefe" d="M85 96.5c-11.11 6.13-26.38 6.76-41.75.47A64.53 64.53 0 0113.84 73a50 50 0 0010.85 6.32c15.87 7.1 31.73 6.61 42.9 0-15.9-11.66-29.4-26.82-39.46-39.2a43.47 43.47 0 01-5.29-6.82c12.16 10.61 31.5 24 38.38 27.79a271.77 271.77 0 01-27-32.34 266.8 266.8 0 0044.47 34.87c.71.38 1.26.7 1.7 1a32.7 32.7 0 001.21-3.51c3.71-12.89-.53-27.54-9.79-39.67C93.25 33.81 106 57.05 100.66 76.51c-.14.53-.29 1-.45 1.55l.19.22c10.59 12.63 7.68 26 6.35 23.5C101 91 90.37 94.33 85 96.5z"></path>'
    RUST = '<path d="M62.271 10.88c-.189.11-.982 1.248-1.763 2.529-1.96 3.217-1.982 3.219-4.615.448-1.713-1.802-2.127-2.132-2.679-2.128-.359.002-.812.124-1.008.271-.195.147-.748 1.317-1.228 2.6-1.099 2.939-1.152 3.034-1.761 3.151-.375.071-1.097-.331-2.828-1.574-1.278-.919-2.532-1.67-2.786-1.67-1.054 0-1.351.576-1.853 3.593-.638 3.836-.616 3.823-4.074 2.252-1.396-.633-2.72-1.152-2.943-1.152-.223 0-.646.24-.939.533-.532.533-.533.535-.388 3.468l.146 2.936-.555.297c-.492.263-.831.231-3.009-.284-2.843-.671-3.443-.653-4.019.122l-.421.566.565 2.421c.31 1.331.609 2.613.665 2.848.055.234-.04.609-.212.832-.284.367-.586.4-3.217.36-4.453-.07-4.706.312-2.866 4.328.585 1.275 1.064 2.433 1.064 2.572 0 .734-.585 1.001-3.098 1.411-1.406.229-2.628.417-2.716.417-.088 0-.352.192-.586.426-.765.765-.548 1.483 1.187 3.932 2.161 3.05 2.157 3.061-1.413 4.427-4.06 1.553-4.142 1.936-1.051 4.868 2.879 2.73 2.882 2.69-.377 4.739-2.469 1.551-2.507 1.588-2.57 2.429-.076 1.023-.058 1.041 2.89 2.842 2.915 1.78 2.915 1.834.054 4.541-3.077 2.91-2.982 3.335 1.081 4.868 3.55 1.339 3.555 1.355 1.39 4.405-1.227 1.729-1.618 2.449-1.618 2.983 0 .999.52 1.254 3.627 1.776 2.617.441 3.2.7 3.2 1.422 0 .148-.48 1.316-1.067 2.594-1.826 3.977-1.618 4.308 2.704 4.308 4.025 0 3.918-.123 3.051 3.507-.654 2.736-.664 3.26-.072 3.851.453.454 1.307.403 3.978-.236 2.04-.487 2.398-.521 2.871-.268l.54.289-.146 2.935c-.145 2.934-.144 2.936.388 3.469.293.293.722.533.952.533.23 0 1.554-.516 2.943-1.147 3.447-1.565 3.425-1.578 4.061 2.246.504 3.031.798 3.594 1.874 3.594.267 0 1.494-.72 2.728-1.6 2.167-1.546 2.729-1.788 3.306-1.421.149.094.727 1.364 1.284 2.822.819 2.144 1.119 2.702 1.575 2.92.868.416 1.405.082 3.445-2.14 2.463-2.683 2.564-2.67 4.575.589 2.221 3.598 2.796 3.59 5.073-.073 1.962-3.156 1.939-3.154 4.591-.384 1.761 1.838 2.136 2.131 2.73 2.131.379 0 .832-.142 1.005-.316.174-.174.75-1.459 1.28-2.855.53-1.397 1.079-2.613 1.221-2.703.561-.357 1.142-.106 3.306 1.43 1.274.905 2.473 1.6 2.758 1.6 1.058 0 1.44-.751 1.88-3.703.376-2.517.452-2.758.947-3.009.487-.247.779-.164 3.063.873 1.389.63 2.713 1.146 2.943 1.146.23 0 .666-.247.967-.549l.549-.548-.151-2.815c-.144-2.688-.131-2.832.298-3.22.441-.399.486-.397 2.952.166 2.986.682 3.543.7 4.104.139.548-.548.542-.668-.208-3.831-.841-3.548-.954-3.422 3.088-3.422 2.755 0 3.062-.039 3.413-.426.586-.648.447-1.39-.732-3.903-.595-1.266-1.078-2.418-1.074-2.56.02-.747.607-1.002 3.32-1.443 1.66-.269 2.902-.581 3.127-.784.754-.681.477-1.567-1.244-3.98-2.157-3.024-2.148-3.053 1.306-4.326 4.136-1.524 4.254-2.032 1.159-4.973-2.867-2.724-2.868-2.709.272-4.637 3.796-2.33 3.802-2.855.067-5.173-3.212-1.993-3.21-1.965-.331-4.699 3.088-2.934 3.004-3.318-1.057-4.871-3.584-1.371-3.595-1.405-1.417-4.394 1.297-1.78 1.618-2.371 1.618-2.981 0-1.066-.478-1.305-3.622-1.813-2.627-.424-3.205-.682-3.205-1.429 0-.142.48-1.285 1.067-2.542 1.149-2.461 1.31-3.446.66-4.035-.349-.316-.817-.361-3.321-.32-2.62.044-2.955.007-3.318-.358-.397-.399-.393-.455.227-3.042.76-3.17.763-3.247.138-3.834-.634-.596-1.03-.586-3.941.099-2.121.5-2.472.533-2.954.275l-.547-.293.151-2.926.152-2.925-.547-.547c-.301-.301-.728-.547-.95-.547-.221 0-1.538.523-2.926 1.161-2.318 1.067-2.567 1.138-3.068.876-.5-.262-.583-.52-1.01-3.127-.493-3.016-.798-3.603-1.869-3.603-.254 0-1.513.755-2.798 1.678-2.11 1.516-2.393 1.659-2.919 1.476-.435-.152-.688-.483-.997-1.306-.229-.606-.667-1.774-.975-2.595-.622-1.656-.969-2.027-1.901-2.027-.52 0-.991.374-2.679 2.127-2.653 2.756-2.663 2.755-4.614-.445-.78-1.279-1.595-2.421-1.812-2.537-.488-.262-1.062-.261-1.511.002m2.418 9.635c2.311 1.645 1.082 5.512-1.752 5.512-2.75 0-4.135-3.313-2.171-5.194 1.108-1.062 2.697-1.191 3.923-.318m-2.906 10.214c1.515.576 2.137.23 5.596-3.104l2.599-2.506 1.1.146c3.45.458 10.312 3.472 14.255 6.261 3.623 2.564 8.438 7.786 10.49 11.377l.439.769-1.944 4.38c-1.07 2.409-1.945 4.633-1.945 4.944 0 .717.47 1.851.923 2.226.191.159 2.006 1.033 4.033 1.942l3.684 1.654.145.937c.187 1.221.212 4.22.042 5.072l-.133.666h-2.103c-2.439 0-2.251-.218-2.383 2.774-.096 2.169-.62 3.368-1.812 4.144-1.942 1.267-5.149 1.037-6.509-.466-.209-.231-.615-1.392-.903-2.581-.841-3.473-1.971-5.423-4.241-7.32-.717-.599-1.303-1.158-1.303-1.243 0-.084.788-.748 1.752-1.473 3.51-2.646 5.528-5.726 5.75-8.777.423-5.819-4.213-11.243-11.109-13.001-1.635-.417-2.333-.43-22.56-.43-11.48 0-20.873-.075-20.873-.166 0-.215 2.551-2.691 4.054-3.933 4.127-3.412 9.488-6.097 15.04-7.531l1.92-.497 2.728 2.766c1.501 1.521 2.972 2.857 3.268 2.97M27.432 48.526c1.257.823 1.772 2.891 1.03 4.134-1.148 1.924-4.056 2.005-5.205.145-1.671-2.702 1.547-6.001 4.175-4.279m74.05.105c3.288 2.005.74 6.937-2.78 5.38-2.35-1.04-2.425-4.252-.127-5.424.959-.489 2.061-.472 2.907.044M37.12 60.907v12.266H26.276l-.43-1.866c-.846-3.675-1.202-7.477-.989-10.591l.149-2.188 3.728-1.672c2.339-1.048 3.843-1.847 4.037-2.144.848-1.293.767-2.217-.423-4.845l-.556-1.227h5.328v12.267m31.22-11.733c2.322.604 3.549 1.833 3.552 3.556.002 1.265-.625 2.059-2.18 2.761-1.101.498-1.276.51-8.219.578l-7.093.068v-7.284h6.355c4.964 0 6.625.07 7.585.321m-2.396 17.602c1.151.32 2.512 1.32 3.21 2.359.733 1.092 1.162 2.512 2.178 7.216.858 3.976 1.41 5.276 2.956 6.968 1.915 2.095 1.471 2.014 11.037 2.014 4.581 0 8.328.073 8.328.163 0 .161-3.155 3.891-3.291 3.891-.039 0-1.687-.345-3.662-.767-5.577-1.191-5.778-1.051-7.058 4.926l-.823 3.84-.743.366c-1.24.612-5.27 1.872-7.359 2.302-3.452.71-7.209.95-10.511.671-5.629-.477-13.083-2.661-13.374-3.92-.062-.267-.437-1.995-.832-3.841-.396-1.846-.877-3.597-1.069-3.891-.923-1.408-1.894-1.495-6.164-.55-1.617.358-3.028.65-3.136.65-.203 0-3.204-3.47-3.204-3.704 0-.073 7.128-.158 15.84-.188l15.84-.054.057-5.627c.04-3.973-.015-5.714-.187-5.92-.192-.232-1.214-.293-4.91-.293H54.4V66.56l5.387.001c2.962.001 5.733.098 6.157.215M41.536 92.365c2.519 1.535 1.311 5.557-1.668 5.554-3.055-.002-4.187-3.987-1.584-5.575.861-.525 2.374-.515 3.252.021m46.126.168c1.235.905 1.646 2.788.881 4.042-2.009 3.295-7.033.676-5.355-2.791.825-1.703 3.018-2.317 4.474-1.251" fill-rule="evenodd"></path>'
    RUBY = '<linearGradient id="ruby-original-a" gradientUnits="userSpaceOnUse" x1="157.08" y1="2382.05" x2="131.682" y2="2426.892" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#FB7655"></stop><stop offset="0" stop-color="#FB7655"></stop><stop offset=".41" stop-color="#E42B1E"></stop><stop offset=".99" stop-color="#900"></stop><stop offset="1" stop-color="#900"></stop></linearGradient><path fill="url(#ruby-original-a)" d="M97.078 83.214L28.34 124.031l89.003-6.04 6.855-89.745z"></path><linearGradient id="ruby-original-b" gradientUnits="userSpaceOnUse" x1="169.731" y1="2419.72" x2="136.998" y2="2441.685" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#871101"></stop><stop offset="0" stop-color="#871101"></stop><stop offset=".99" stop-color="#911209"></stop><stop offset="1" stop-color="#911209"></stop></linearGradient><path fill="url(#ruby-original-b)" d="M117.488 117.93l-7.649-52.799-20.837 27.514z"></path><linearGradient id="ruby-original-c" gradientUnits="userSpaceOnUse" x1="143.542" y1="2380.69" x2="110.81" y2="2402.655" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#871101"></stop><stop offset="0" stop-color="#871101"></stop><stop offset=".99" stop-color="#911209"></stop><stop offset="1" stop-color="#911209"></stop></linearGradient><path fill="url(#ruby-original-c)" d="M117.592 117.93l-56.044-4.399-32.91 10.385z"></path><linearGradient id="ruby-original-d" gradientUnits="userSpaceOnUse" x1="74.817" y1="2435.622" x2="79.891" y2="2402.644" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#fff"></stop><stop offset="0" stop-color="#fff"></stop><stop offset=".23" stop-color="#E57252"></stop><stop offset=".46" stop-color="#DE3B20"></stop><stop offset=".99" stop-color="#A60003"></stop><stop offset="1" stop-color="#A60003"></stop></linearGradient><path fill="url(#ruby-original-d)" d="M28.717 123.928l14.001-45.867-30.81 6.588z"></path><linearGradient id="ruby-original-e" gradientUnits="userSpaceOnUse" x1="109.719" y1="2466.413" x2="111.589" y2="2432.757" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#fff"></stop><stop offset="0" stop-color="#fff"></stop><stop offset=".23" stop-color="#E4714E"></stop><stop offset=".56" stop-color="#BE1A0D"></stop><stop offset=".99" stop-color="#A80D00"></stop><stop offset="1" stop-color="#A80D00"></stop></linearGradient><path fill="url(#ruby-original-e)" d="M88.996 92.797l-12.882-50.46-36.866 34.558z"></path><linearGradient id="ruby-original-f" gradientUnits="userSpaceOnUse" x1="140.691" y1="2497.523" x2="146.289" y2="2473.401" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#fff"></stop><stop offset="0" stop-color="#fff"></stop><stop offset=".18" stop-color="#E46342"></stop><stop offset=".4" stop-color="#C82410"></stop><stop offset=".99" stop-color="#A80D00"></stop><stop offset="1" stop-color="#A80D00"></stop></linearGradient><path fill="url(#ruby-original-f)" d="M121.275 43.047L86.426 14.585l-9.704 31.373z"></path><linearGradient id="ruby-original-g" gradientUnits="userSpaceOnUse" x1="123.6" y1="2506.018" x2="147.719" y2="2518.077" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#fff"></stop><stop offset="0" stop-color="#fff"></stop><stop offset=".54" stop-color="#C81F11"></stop><stop offset=".99" stop-color="#BF0905"></stop><stop offset="1" stop-color="#BF0905"></stop></linearGradient><path fill="url(#ruby-original-g)" d="M104.978 4.437L84.481 15.764 71.551 4.285z"></path><linearGradient id="ruby-original-h" gradientUnits="userSpaceOnUse" x1="53.674" y1="2444.028" x2="55.66" y2="2424.153" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#fff"></stop><stop offset="0" stop-color="#fff"></stop><stop offset=".31" stop-color="#DE4024"></stop><stop offset=".99" stop-color="#BF190B"></stop><stop offset="1" stop-color="#BF190B"></stop></linearGradient><path fill="url(#ruby-original-h)" d="M3.802 100.034l8.586-15.659L5.442 65.72z"></path><path fill="#fff" d="M4.981 65.131l6.987 19.821 30.365-6.812L77 45.922l9.783-31.075L71.38 3.969l-26.19 9.802c-8.252 7.675-24.263 22.86-24.84 23.146-.573.291-10.575 19.195-15.369 28.214z"></path><linearGradient id="ruby-original-i" gradientUnits="userSpaceOnUse" x1="40.026" y1="2418.781" x2="133.345" y2="2514.739" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#BD0012"></stop><stop offset="0" stop-color="#BD0012"></stop><stop offset=".07" stop-color="#fff"></stop><stop offset=".17" stop-color="#fff"></stop><stop offset=".27" stop-color="#C82F1C"></stop><stop offset=".33" stop-color="#820C01"></stop><stop offset=".46" stop-color="#A31601"></stop><stop offset=".72" stop-color="#B31301"></stop><stop offset=".99" stop-color="#E82609"></stop><stop offset="1" stop-color="#E82609"></stop></linearGradient><path fill="url(#ruby-original-i)" d="M29.519 29.521c17.882-17.73 40.937-28.207 49.785-19.28 8.843 8.926-.534 30.62-18.418 48.345-17.884 17.725-40.653 28.779-49.493 19.852-8.849-8.92.242-31.191 18.126-48.917z"></path><linearGradient id="ruby-original-j" gradientUnits="userSpaceOnUse" x1="111.507" y1="2409.102" x2="83.398" y2="2416.039" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#8C0C01"></stop><stop offset="0" stop-color="#8C0C01"></stop><stop offset=".54" stop-color="#990C00"></stop><stop offset=".99" stop-color="#A80D0E"></stop><stop offset="1" stop-color="#A80D0E"></stop></linearGradient><path fill="url(#ruby-original-j)" d="M28.717 123.909l13.89-46.012 46.135 14.82c-16.68 15.642-35.233 28.865-60.025 31.192z"></path><linearGradient id="ruby-original-k" gradientUnits="userSpaceOnUse" x1="159.785" y1="2442.837" x2="134.814" y2="2465.217" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#7E110B"></stop><stop offset="0" stop-color="#7E110B"></stop><stop offset=".99" stop-color="#9E0C00"></stop><stop offset="1" stop-color="#9E0C00"></stop></linearGradient><path fill="url(#ruby-original-k)" d="M77.062 45.831l11.844 46.911c13.934-14.65 26.439-30.401 32.563-49.883l-44.407 2.972z"></path><linearGradient id="ruby-original-l" gradientUnits="userSpaceOnUse" x1="168.959" y1="2483.901" x2="156.521" y2="2497.199" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#79130D"></stop><stop offset="0" stop-color="#79130D"></stop><stop offset=".99" stop-color="#9E120B"></stop><stop offset="1" stop-color="#9E120B"></stop></linearGradient><path fill="url(#ruby-original-l)" d="M121.348 43.097c4.74-14.305 5.833-34.825-16.517-38.635l-18.339 10.13 34.856 28.505z"></path><path fill="#9E1209" d="M3.802 99.828c.656 23.608 17.689 23.959 24.945 24.167l-16.759-39.14-8.186 14.973z"></path><radialGradient id="ruby-original-m" cx="138.703" cy="2464.789" r="30.601" gradientTransform="matrix(1 0 0 -1 -47.5 2517)" gradientUnits="userSpaceOnUse"><stop offset="0" stop-color="#A80D00"></stop><stop offset="0" stop-color="#A80D00"></stop><stop offset=".99" stop-color="#7E0E08"></stop><stop offset="1" stop-color="#7E0E08"></stop></radialGradient><path fill="url(#ruby-original-m)" d="M77.128 45.904c10.708 6.581 32.286 19.798 32.723 20.041.68.383 9.304-14.542 11.261-22.976l-43.984 2.935z"></path><radialGradient id="ruby-original-n" cx="96.325" cy="2424.465" r="40.679" gradientTransform="matrix(1 0 0 -1 -47.5 2517)" gradientUnits="userSpaceOnUse"><stop offset="0" stop-color="#A30C00"></stop><stop offset="0" stop-color="#A30C00"></stop><stop offset=".99" stop-color="#800E08"></stop><stop offset="1" stop-color="#800E08"></stop></radialGradient><path fill="url(#ruby-original-n)" d="M42.589 77.897l18.57 35.828c10.98-5.955 19.579-13.211 27.454-20.983L42.589 77.897z"></path><linearGradient id="ruby-original-o" gradientUnits="userSpaceOnUse" x1="67.509" y1="2393.115" x2="57.373" y2="2427.506" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#8B2114"></stop><stop offset="0" stop-color="#8B2114"></stop><stop offset=".43" stop-color="#9E100A"></stop><stop offset=".99" stop-color="#B3100C"></stop><stop offset="1" stop-color="#B3100C"></stop></linearGradient><path fill="url(#ruby-original-o)" d="M11.914 84.904l-2.631 31.331c4.964 6.781 11.794 7.371 18.96 6.842-5.184-12.9-15.538-38.696-16.329-38.173z"></path><linearGradient id="ruby-original-p" gradientUnits="userSpaceOnUse" x1="145.272" y1="2507.076" x2="167.996" y2="2497.045" gradientTransform="matrix(1 0 0 -1 -47.5 2517)"><stop offset="0" stop-color="#B31000"></stop><stop offset="0" stop-color="#B31000"></stop><stop offset=".44" stop-color="#910F08"></stop><stop offset=".99" stop-color="#791C12"></stop><stop offset="1" stop-color="#791C12"></stop></linearGradient><path fill="url(#ruby-original-p)" d="M86.384 14.67l36.891 5.177c-1.969-8.343-8.015-13.727-18.32-15.41L86.384 14.67z"></path>'
    KOTLIN = '<linearGradient id="kotlin-original-a" gradientUnits="userSpaceOnUse" x1="-11.899" y1="48.694" x2="40.299" y2="-8.322"><stop offset="0" stop-color="#1c93c1"></stop><stop offset=".163" stop-color="#2391c0"></stop><stop offset=".404" stop-color="#378bbe"></stop><stop offset=".696" stop-color="#587eb9"></stop><stop offset=".995" stop-color="#7f6cb1"></stop></linearGradient><path fill="url(#kotlin-original-a)" d="M0 0h65.4L0 64.4z"></path><linearGradient id="kotlin-original-b" gradientUnits="userSpaceOnUse" x1="43.553" y1="149.174" x2="95.988" y2="94.876"><stop offset="0" stop-color="#1c93c1"></stop><stop offset=".216" stop-color="#2d8ebf"></stop><stop offset=".64" stop-color="#587eb9"></stop><stop offset=".995" stop-color="#7f6cb1"></stop></linearGradient><path fill="url(#kotlin-original-b)" d="M128 128L64.6 62.6 0 128z"></path><linearGradient id="kotlin-original-c" gradientUnits="userSpaceOnUse" x1="3.24" y1="95.249" x2="92.481" y2="2.116"><stop offset="0" stop-color="#c757a7"></stop><stop offset=".046" stop-color="#ca5a9e"></stop><stop offset=".241" stop-color="#d66779"></stop><stop offset=".428" stop-color="#e17357"></stop><stop offset=".6" stop-color="#e97c3a"></stop><stop offset=".756" stop-color="#ef8324"></stop><stop offset=".888" stop-color="#f28817"></stop><stop offset=".982" stop-color="#f48912"></stop></linearGradient><path fill="url(#kotlin-original-c)" d="M0 128L128 0H64.6L0 63.7z"></path>'
    DART = '<path fill="#00c4b3" d="M35.2 34.9l-8.3-8.3v59.7l.1 2.8c0 1.3.2 2.8.7 4.3l65.6 23.1 16.3-7.2-74.4-74.4z"></path><path d="M27.7 93.4zm81.9 15.9l-16.3 7.2-65.4-23.1c1.3 4.8 4 10.1 7 13.2l21.3 21.2 47.6.1 5.8-18.6z" fill="#22d3c5"></path><path fill="#0075c9" d="M1.7 65.1C-.4 67.3.7 72 4 75.5l14.7 14.8 9.2 3.3c-.3-1.5-.7-3-.7-4.3l-.1-2.8-.2-59.8m82.7 82.6l7.2-16.4-23-65.6c-1.5-.3-3-.6-4.3-.7l-2.9-.1-59.6.1"></path><path d="M93.6 27.3c.2 0 .2 0 0 0 .2 0 .2 0 0 0zm16 82l17.7-5.8V54.8l-20.4-20.5c-3-3-8.3-5.8-13.2-7l23.1 65.6" fill="#00a8e1"></path><path fill="#00c4b3" d="M90.5 18.2L75.7 3.5c-3.4-3.4-8-4.4-10.4-2.3L26.9 26.6h59.5l2.9.1c1.3 0 2.8.2 4.3.7l-3.1-9.2z"></path>'
    PHP = '<defs><path id="php-original-a" d="M64.026 96.076c33.676 0 60.976-14.361 60.976-32.076s-27.3-32.075-60.976-32.075S3.051 46.285 3.051 64s27.3 32.076 60.975 32.076"></path></defs><defs><path id="php-original-c" d="M2.998 31.924h122.004v64.1H2.998z"></path></defs><clipPath id="php-original-b"><use xlink:href="#php-original-a" overflow="visible"></use></clipPath><clipPath id="php-original-d" clip-path="url(#php-original-b)"><use xlink:href="#php-original-c" overflow="visible"></use></clipPath><g clip-path="url(#php-original-d)"><defs><path id="php-original-e" d="M2.998 31.924h122.004v64.1H2.998z"></path></defs><clipPath id="php-original-f"><use xlink:href="#php-original-e" overflow="visible"></use></clipPath><g clip-path="url(#php-original-f)"><image overflow="visible" width="1160" height="609" xlink:href="data:image/jpeg;base64,/9j/4AAQSkZJRgABAgECqAKoAAD/7AARRHVja3kAAQAEAAAAHgAA/+4AIUFkb2JlAGTAAAAAAQMA EAMCAwYAABe6AAAnEwAALd7/2wCEABALCwsMCxAMDBAXDw0PFxsUEBAUGx8XFxcXFx8eFxoaGhoX Hh4jJSclIx4vLzMzLy9AQEBAQEBAQEBAQEBAQEABEQ8PERMRFRISFRQRFBEUGhQWFhQaJhoaHBoa JjAjHh4eHiMwKy4nJycuKzU1MDA1NUBAP0BAQEBAQEBAQEBAQP/CABEIAmYEkwMBIgACEQEDEQH/ xAClAAEBAQADAQEAAAAAAAAAAAAAAQIEBgcFAwEBAQEBAAAAAAAAAAAAAAAAAAEDAhABAQEAAQAJ BQEBAQADAAAAAAERECBgAzM0BRUmBzBAUCUWBgInEhMEEQAAAwUIAgICAgMBAAAAAAAAAQIQkaFD NHCx0XKyA3MEQkQRohITITFBYXEVEgABBQEBAAIDAQAAAAAAAAAAYAExQUKBQzACQJAygv/aAAwD AQACEQMRAAAA775/6B5h1PoOA6nPcCnPcAc9wKc68BXPcCnOcEnOcEc9wRznBVznBHOcEc68Ac9w RznBHOcEc5wRznBWc5wRznBHOcEc5wRznBHOcEc5wRznBLznBRznBHOcEc5wYc5wRznBHOcEc5wU c6cEc5wRznBLznAHOcFHOcAc+cEc5wBznBkc+cEvOcAc5wBznAkc9wIc9wIc+cCHPfPh9CfPkv0J 8+H0J8/J9HPz4fQz8/MfRz87K/SfLh9V8ofVfKH1Xyh9V8ofVfKH1Xyh9V8ofVfKH1Xyh9V8ofVf KH1Xyh9V8ofV7n5v387iEx5j6d5j1FOoFKAqSgFFAAAWFEoAABQAAAAAAAAAAAAAQABFEWAKEJRF gBFRARYqURZEWCUSVElLJRlZEmoZmoZmoZmpLmahiahjP6ZPzz+mY/PP6ZXIAAAAAAAAAAAAHf8A oHfzuITHmXpvmXUF7gIUBQoAALZKAAAUAACCkURVSgAAAABFEVEUQAKAAAEJRFEChCUQCURZCUsB FkJRJRJqRJSyUZWRJqGZqGZqGZqS5zuGM7hjO4flN5jGd5WAAAAAAAAAAAAd/wCgd/O4hMeZ+meZ 9wXrmUoUlAAWyUAAoAAEKJSgAAAAAAAAAAAAAAIoiyAUAAIAiiBQiLACKiSiLFSokogJNSJNRZKM rIk1DM1DM1DM1JcTUMTeTGf0yfnneYxNZUAAAAAAAAAAB3/oHfzuITHmfpnmncDvkUACikACgABU lKAAABAoAAAAAAAAAAAAJQAAIqIAFACEoiwCVKIBKiAixUqJKICLIkpZKMrIk1DM1DOd5MzUlzne TOd5MZ/TBjO8y4WAAAAAAAAAADv/AEDv53EJjzT0vzTvlTuAC2SgAFACoFAABYAAAAFCIBUFQVBU FgLBQohRAAAASgARRBAKAERRAoRFglEWRFipRJUQEmpElLJYSakZmoZmoZmoZzuS4zvJnO8mMfpm Pzz+mVyAAAAAAAAAB3/oHfzuITHmvpXmvfIdwWwAKAFSUoAALAACWkEqAAAAKBAAAAAlAAWCwKlC WUAAIBQIqIFACEogAliwAiyIsVKJKiAkqJKXKwk1IzNQzNQznUjM1FxNZM4/TJ+ef0xGJrKgAAAA AAAAO/8AQO/ncQmPNfSvNe+VNOQAoApAoALABCkssAAALAAAAAQAAAFAAAAACVYKgpFogAJQEoiy AUIAiwCWLACLIixUqJKJKJKiSxZNQmdSMzUMzUMzUlxNQxneTGf0wYx+mZcLAAAAAAAAB3/oHfzu ITHm3pPm2nId8hQIpQAIFAJZZYAAWAABYAURRFsuWhFEUSaGWhlRFWQAASgAAoRYFEoAASgRZAKA ERYBLFgBBEWKlRJYJRlZLJqElhJqRmWEzqGZqS4zvJnO8n55/TEYzvKwAAAAAAADv/QO/ncQmPNv SfNtOQ05FQKACwAKQQALAAQKKJali0iiLSKIoiiKIoiiTQyok0MqMtQiywAJQAUIqUCUAJQIqIFA CIsAliwSiCIsWLIiwiwk1JZLCTUjM1DM1DE1Jc53kzneTGP0yfnneZcLAAAAAAAB3/oHfzuITHm3 pPm2nIunAUAFgCFgAWAAgtSrLFpFBRLSxRLRFEURpGWhlqEVUUZahFGWoSaiSaGZoZVZABKAEoFE oAQCpRBKAERYBLFglRJRAsWRASVEliyahM6kZmoZmskzqS4moYzvJjH6YjGd5WAAAAAAAd/6B387 iEx5t6T5vpyGvAAWCAWABYAFhbLKpKpKqyqRQVEtEURaRRFLFEUkmhlRFGVEmoRVZUZUmVGZqEWW BAKChFQtEBAKlEEoCVECpUQEWQlipUSUSUZWSyahJZEzqGZqGZrJmazLnO8mM7yfnneZcLAAAAAA B3/oHfzuITHm/pHm+vAachYIBYAFgILRUqqSqsqkqxKpKpFqxRLRFEURoZaGWoRRFGWoSaGVGVJl RlVZUZUmZqEmoQWBKAEqwtEBALFkAoQlEEqUQEERYqWRFhFkSUuVhJZGZqGZqGc6kuc6hjO8n55/ TEYzvKwAAAAADv8A0Dv53EJjzf0jzfXgNeBALAAsCwqFVShVWVYlUKUoKJaiLSKI0IoiiKIok0Mt Qk0MqMtSpNQk1DLUTKjKqzNQk1EzNQgsCUBYlWFogIBYsgFCIsAliwSokogWSokokslk1CSyJnUM zUMywzneZc51DGP0wYx+mZcAAAAAAd/6B387iEx5v6R5vrwGvEFgAWBYqyyqKqiirEqhSlBbEWkU FLLRFEWmWhloZaGVEUZUSahJomZoZmoSalZahlSZmoZVWZqJlRlZYEoAS1KoQEoEEAoRABLARZEW KlkRYRZElLlYSWRmahmayZmpLjO8mcbyYzvEYmsqAAAAA7/0Dv53EJjzf0jzfXhDbMALAsVZVBVV VJVhVWVQqFUlVZVJVItIqJaIoiiKIoijLUJNDKqyoyoypMzUJNSsqMzUTKwk1KzNRJKMrLAlAWJa llACVKIJQhKIJUsEogiBZKiSiSyWTUJLImdQzNQxNSXOdQxneTGP0xGM7wsAAAAA7/0Dv53EJjzb 0nzbbgNcwsBFlFVVVVUKhVVVJVhVWVSWgqCgoloi0y0WKIok0MtEyoy1CTUJNQk1Ky1DKjM1Eyoz NSszUJKTM1KzNRMrCCwJQlFAlACWLIBQiLAJYBLISxUsiLCLIksWTUJnUjM1kmdQznUlxNZM4/TB jG8y4WAAAADv/QO/ncQmPNvSfNtsw2zAFsVZVVVUWWFVVUVYlVVUlqFBVJaWWiLSKiLTNoijLQyo k0rKjLUMzRMzUJNQyqszUMqTM1DM1KzNQksSTUrM1EzNQgsCUJahaICUCCAVKiCVLBKJKiBZKiSj KyWSwk1IzLCZ1DOdSXOdZM53k/PO8x+c1lQAAAHf+gd/O4hMebek+bbZhtmFlsstspbKqrCqqqKs KqyqKsFCqpQVBaRaZtEWmWhloZaGVGWoSahJqEmpWWoZmokmoZmpWVGZqJmahmalZmomZqGVlSWJ JZYEoCxLUsoASpZAKERYBLARZEWKlkQElRJYslhJZGZqGc6kuc6hjO8mMfpiMY/TCwAAADv/AEDv 53EJjzb0nzbbMN8lWVVFVVWFVVUVYVVFFWFUVVKgtJaJaJaJaItjLQy0MtDKjLUrLUMtQzNDM1DL UrM1EzNQzNSpKMzUTM1DM1KzNRMzUrM1DM1EyssCUJagolCUCCUBLICWLBLISxUqJLBLCSpcrCZ1 IzNZJnUM51JcTWTGd4MZ3iXIAAAHf+gd/O4hMebek+bbZi75KsqqqrCqUqqsUq0oqwqiqqrCqS0F pLUS0stEWmWhloZaGVGWoZahmaJmalZmhmahmalZmoZmomZqVmahmaiZmoZmpWZqJmalZmomZqEF gSrEtSygBLFkAoRAqVEBBECyVElElkslhJqRmWEzrJM6kuM7yZxvJ+ed5jEsUAAB3/oHfzuITHm3 pPm2+SrrwqjUq2yxbKts1CqqqKsKoqqqwqiqKsS2rKpLRLRGkZtEUZaGWoZalZm4ZmoZahmalmZq GZqGZqVmahmaiZmpWZqGZrKSWVJYZmomZqVmaiZWWBKEtSgShKlEEoQlgEsBFkRYqWRASWRFiyWR M6hmWGZrMuc7yZxvJjG8xjOsqAAA7/0Dv53EJjzb0nzfbNV2zVVtli2VbVFWFVWpYtlLVVVhVFUV qWVRVC2JaJaJaIoiiTQy1DLUMzcMzUrM1DM1DM1EzNSszUMzUrM1lJLDM1KzNQzNRMyypLEmdSsz UTM1kBAVZZQlCAWCAVKiCVLBLISxUshLCLIksWSwk1mJLDMsjOdRcTWTGP0xGMbysAAA7/0Dv53E Jjzf0jzfbO1ds1VbZYtlW1YVVtlLVhVW2WLVFUVqVVFUVYLSWiWpZaI0MtDLUMtQy1KzNEzNQzNQ zNSs53kzNSzM1DM1kmdSszUTM1kmdSszUTM1mpLDM1EzNSszWUgsCUJVJQAlSyAUIgVLIAkqIFks hLCSpZLCSyMywmdZJnWZc51DGN5MY3mXIAAHf+gd/O4hMecej+cbZ2y7Z2rK1KWyrasNSrbNQqls 0qrDUpasNSraoqwrRLbLKpLaRoZtEahloZmhhqGZqGZvNmZqGZqGc7zWZqGZrNmZqGZrJM6lZmom ZZWZqGZrKSWVmaiZms1JYkllgShLUoEoSpYBKERYBLAQRAsESWEWSyWElkTOoZlhmWS5zrJnG8mM bxLkAADv/QO/ncQmPOfRvOds7ZduLZYupVtli1ValLZYupVtmoalLVLZqVVLVhWhVlVRaFWItItM tDM0MzQzNQzNSszUM53KxNRM53kzNSsTUMzWbMzUMyyszWUksMzUrMsSSyszWUksqSxMzWQABZZQ lCUCCUISwCWAiyIFiyICSyWSwk1IzLCZ1kmdZlzneTON4MZ1mXCwAAd/6B387iEx5z6N5ztnau3F ssWzStSxbKt1KWyxdSrbNRbNDUstqlqxao1NSqpasKoWktqxURRJoYm5ZhqGZqGJvNZmoYmpZnOo ZzvNZmsmZqWZzqGZYZms2SWGZrNSWJmalZlhM6iZllSWJBYEqyyhKEAsEAqWQEsAlkJYqWRASWRF iyWRmaySWGZZLnOsmc6yYzrMYligAO/9A7+dxCY869F862ztl24tmotlW2ai2VbZqGpS6lW2ai2a LZZbqUupqGpoallupoVYaUVZS0jQy0MzcMNQzNQxN5szNQxNZrM1kzNZszNZMzWazNRM51KzLDM1 mszUTMsrM1DMsSZ1KzLEksqZ1lILAlCWpZQAlggFSoglSwSyEsVLIiwkshLFksiZ1DMsMyyXOdZM 53k/PO8RiWKAA7/0Dv53EJjzr0XzrbO2a24tli2VbqWLZpbZYus6LZZdWU1ZY1ZVupqLZotmpbZo tWLWhVlaUW2ItMtDM0MTeaxN5M53DGd5szneaznUM51LM51DE1mpnWSZ1myZ1kksrMsSZ1KzLDM1 lJLKmdRMyypLEyssCUJaFCAlSyAUIglSwSyEsVLISwgiSxZLCSyMzWSZ1mXM1kznWTGN4jOdZUAB 3/oHfzuITHnXovnW2d1LrxbKt1KWyxdSrbLGrKt1LF1KXWdRdZ0t1LGrKas1LdZ0XUsas1LapdTU K0GiybkYm5ZiahjO81nOoZzrNmZrJnOpWc6zUzrKZms1mWGZZZmayZms1M6yklhmazUliZms1JYk zrNSWJJZYEoS0KEBKlkAoRAqWQlRJYqWQlglkSWLJYSWRM6yTOsyzOsmc6yYzrMYzrKgAO/9A7+d xCY869F862z1ZdeLZpbZYtlXVlLrOotmltljVlNWWXVlNazqLrOi6zqXWs6i6zo1ZqW6mi6ljWpq VWiTcMTUMZ3izOd5MZ3ms51msyxM51mszWTMssznWamdZJnWbJnWSZ1KzLEzLCZ1mpLEmdZqSxJn UrMsSSywJQloUICVLIBQiBUsgIksVLISwSyJLFksJLIksM51Jc51kznWTGd4jGdZUAB3/oHfzuIT Hnfonne2dsuvF1nS2yxbKurKXWdRdZ0tssaspqyy6sprWdRdZ0assutZ1F1nRrWdS61nRrWdRrWd S63jZrNkYzrNZzrNmc6yZzrNZzrNZliZzrNTOsmZZZnOskzrNTOs2TOskllZliZlhJZWZYkllZli SWVmWJJZYEoS0KEBKlkAoRAqWQESWKlkJYJZEliyWElkSWGZZGc6yuc6yZxvEYzrKgAO/wDQO/nc QmPOvRfOts9WXXi6zV1ZYtlXVlLrOous1dWWNWU1ZZdWU1rNjWs6LrOpdazTWs6jWs2XesaNazY/ S40u7mxqSFzcjFzZJckzc1M6zWZcpM6zWZYZlzZM6zUzrJM6zZM6ySXNSWJmWEzrNSWJM6zUliTO s1JYkllgShLQoQEqWQChECpZARJYqWQlglkSWLJYSWRJcklzLM6yZzrJjOsxjOsqAA7/ANA7+dxC Y869F862z1c614tlW6zotljVzpbZY1ZV1c6i6zous6i6zpdXOo1c6NazZdazo1c6jWsal1rGjdzY 3rFN3FXTIuURlBm5qRCZubGbDMuamdZqZsSZsrMsMyyzMsJmypnWUksJmypLEmbKksSZsqSxJLLA lCWhQgJUsgFCIFSyBISxUshLBLIksWSwksiZ1kmdZlmdZM51kxnWIznWVAAd/wCgd/O4hMedei+d bZ2y7caudRbKurnUWyrqyxdZ0Wyy6spq51GrnS6udRq50a1jUutY0bubGtY0bubLrWKbuKbZsVBZ IWSUiDNzUiJJc1IhM2WTNhM2VM2EzZZM6ySWVmWJM6zUlhM2JJZUzYkllSWJIWBKEtChASpZAKEQ SpYJZCWKlkJYQRJYslhJZEzYTOsyzNhnOsmMbxGc6yoADv8A0Dv53EJjzr0XzrbO2Xbi2WNWVbZY 1c6W6zY1c6NXNXWs2NXOjVzqXVzo1c6jVzTdzZd3NN3NjWsU3cpds01cU0zDUgskLJKREkQZuaZu RmyyZsJmypmxJEqS5GbKkRJLKmbCSxJnWaksSSypmxILAlCVSUAJYIBUsgJUsEshLFSyECSyEsWS yJnWSSwzLJc51kmNZM41mMSxQAHf+gd/O4hMec+jec7Z3Wbtxq51Fsq6ubGrKurmmrLGrnS3WbGr nRq5sbuautZsa1jRq5q7ubG7im7ixu4ptku2UaZGpJWpIiIWSCJTKDNlkiEiVJcjNlkiElhM2WSX JJZUlyjNlSWEiJJZZJYQWBKssoShALBASpYBLAJZCWKlkQElkILJZEzYSXJJZLnOsmc6yYzrEZli gAO/9A7+dxCY859G842z1ZduLZYus1dWWLrNXVzTVljVzpbZY1c01rNNXOpdXNN3Go1c01rFl3cU 3cU2zY0g0yNMjUgsgskqyQuUESxEJEpmwkRJLmmbCS5pmxJLkZsqSxJEqSxJLKkuUZsAAFiWiUJQ IJQhLAJYBCIFgiAkslksESJLCZ1kmdZlzLkmNZMZ1mXAAAHf+gd/O4hJ8Q65U6igoqgBQUFEUCiq CgoKQoAUFABQAAAIAEAgAQEFQEBAQIgICFICABAkBBSAAEAoCkQAAKAEQAAEAEQALAQEBBEgIEgQ AAD64fTB/9oACAECAAEFAP8AljGRkZGRkZGRkZGRkZGRkZGRkZGRkZGRkZGRkZGRkZGRkZGRkZGR kZGMYxjPu/8AnqV/z1K/46lf8dSv+OpX/HUr/jqV2fUrs+pXZ9Suz6ldl1K7LqV2XUrsupXZdSux 6ldj1K7HqV2PUrsepXY9Sux6ldj1K7HqV2PUrsepXY9Sux6ldj1K7HqV2PUrsepXY9Sux6ldj1K7 HqV/z1K//9oACAEDAAEFAL9HWta1rWta1rWta1rWta1rW/iL1KvUq9Sr1KvUr/rqV/11K/66lf8A XUr/AK6lf9dSv++pX/fUr/vqV/31K/76ldp1K7TqV2nUrtOpXadSu06ldp1K7TqV2nUrtOpXadSu 06ldp1K7TqV2nUrtOpXadSu06ldp1K7TqV2nUr/rqV//2gAIAQEAAQUA7S2dn675u9d83eu+bvXf N3rvm713zd655u9d83euebvXPN3rnm71zzd655u9c82euebPXPNnrnmz1zzZ655s9c82et+bPXPN nrnmz1zzZ635s9b82et+bPW/Nnrfmz1vzZ635s9b82et+bPW/Nnrfmz1vzZ635s9b82et+bPW/Nn rfmz1vzZ635s9b82et+bPW/Nnrfmz1vzZ635s9b82et+bPW/Nnrfmz1vzZ635s9b82et+bPW/Nnr fmz1vzZ635s9b82et+bPXPNnrnmz1zzZ635s9c82euebPXPNnrnmz1zzZ655s9c82euebvXPN3rn m71zzd655u9d83euebvXfN3rvm713zd675u9d83eu+bvXfOHr3nD17zh695w9e84evecPXvOXr/n L1/zl6/5y9f85f0HnT+g86f0HnT+g86f0Pnb+h87f0Xnb+i88f0Xnj+j88f0fnq/6Tz1f9J58v8A pfPn9N5+v+n8/X/T/wCgf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf 1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1H+gf1 H+gf1H+gf1H+gf4bzH/93mH/AOR2vddZs6FixeLFixYsWfe/HHgna911ns6FixZxYsWKs+9+OPBO 17rrRZ0KsWcWLFixZ958ceCdr3XWm81VWKsWLFiz7v448E7XuutVnNVVirFixYs+6+OPBO17rrXY vFWKqxYsVYs+5+OPBO17rrZV4sWKqxYsVZ9z8ceCdr3XW2rxVi8WLFixZ9x8ceCdr3XW68VVVVix YsWfb/HHgna911svFXiqqqsWKs+3+OPBO17rrfV4qqqxYsVZ9t8ceCdr3XXCrxVVVixViz7X448E 7XuuuFXiqqqsWKs+1+OPBO17rrfeKvFVVWLFirPtPjjwTte665VeKqqsWKsX7P448E7XuvxOMYxn UWrxVVVWLFWfZ/HHgna91+CzjGMYxjGMYxnGM4z87eKvFVVWLFWL9l8ceCdr3X32cYxnGMYxnSxj GM4xjOM/OVeKqqsWKs+y+OPBO17v73GMYzpYxjGMYxnSxjGc5+XvFXiqqqsVYv2Pxx4J2vd/d4xn RzoYxjGMYxjGdDOjnGfmLxV4qqqxYqz7H448E7Xu/uM4zoZzjGfYYxnOdDPzNXiqqqsVV+w+OPBO 17v7bPoYzoYxn1MYzoYz85eKvFVVVYqz7D448E7Xu/u8ZzjOc5xjGMYxjGM5znGc4z83VVVVVWKq /X+OPBO17v7nOlnGMYz62MYzjOM6GfmLxV4qqqrFWfX+OPBO17v7fOc6GM6OM+pjOjjOhnOfmaqq qqqqv1vjjwTte7+1zp50MZzjGMYxjGMYxjGc4zoZ0M/MXirxVVVVV+t8ceCdr3f3OM5znGMYxjGM 6WMYxjGMZ0s6WflLxVVVVVVV+t8ceCdr3f3Gc5zjGMZzjGM6OMYxnOMYxnOc5+ZvFXiqqqqr9X44 8E7Xu/ts5zjGM6GM6GMYxjGMYxnQxnOMYzjOc/LXmrxVVVVV+r8ceCdr3f2udHOhjOcYxjGMYxjG MYxjGM5xnOM/OXiqqqqqqr9T448E7Xu/s8+jnOMYxjPrYxjGM5z6Ofl6vFVVVVX6nxx4J2vd/ZZ9 DOcYxnRxjGMYxjGMYzo4xjOc+hn5Oqqqqqqqqr9T448E7Xu/u84zo4xjGMYxjGMYxjGMZ0c4z89e KvFVVVVX6fxx4J2vd/d50MYxjGMYxjGMYxjGMYxjGM6GM5z8zV4q8VVVVVfp/HHgna9391nOMZxj GcYxjGMYxjGMZxjGcYxnOfnavFVVVVVV+n8ceCdr3f3GdDGcYznGMYxjGMYxjGMYxnOcYzoZ+cvF Xiqqqqr9L448E7Xu/qz6uM6OMYxjOMYxjGMYxjGMYxnRxnOfSv5CrxV4qqqqq/S+OPBO17v6k+rn RxjGMYxjGMYxjGMYxjOMYxnSz6t/H1VVVVVVVVfpfHHgna939xnRxjOMYxjGMYxjGMYxjOMYzjGd LPzt4qqqqqqv0vjjwTte76c+wzo4znGMYxjGMYxjGMYxjGMYznGdHPsL+Lq8VeKqqqqv0fjjwTte 75nTn1JOjjOM4xjGMYxjGMYxjGMYxjGMYzjGdGz6l6dX8NVVV4qqqqq/R+OPBO17vpT6k5n0M4xn OMYxjGMYxjGMYxnGMZxnOdGz69/GXiqqqqqq/R+OPBO17ridOczozmdPOMZzjGMYxjGMYxjGMYxj GcYxn0bPp3i9G/h6vFVVVVVV+j8ceCdr3XRnM4nRnQnTk4xnGMYxjGMYxjGMYxjGMYxjOcZ9G9C9 O838VV4q8VVVVX6Pxx4J2vdp0ZxOZxOjOjOJOMZxnGM4xnGMYxjGMYxjGMYxnGcZ9K9G8Xm8XoVf w1XirxVVVVV+h8ceCdr3fQnM5nQicROhOJ0cYzjGcYxjGMYxjGMYxjGMYs4s6N4vQvNXo3i838TV VVVVVVVV+h8ceCdr3adCcTmcTozoToToSJOM4xjGMYxjGMYxjGMYzjOLFnNnRvQvRvF5vF6F/D1e Kqqqqqr9D448E7Xu5zOZxETmJxOhE4iJxIkSc4xjGMYxjGMYxjGMYxjOhZxeLxeKvNXir0KvRvNX 8LV4qqqqqqv0PjjwTte75nM5nE6M6E4iJxESc4xjGMYxjGMYxjGMYxjGM6F4vF4vF6F6N4vN5v4m rxV4qqqqv0PjjwTte7nQnE5nMTiJzETicTiToSM4xjGMYxjGMYxjGMYzjFnQs4qrxVXmrxV5vN4v Qv4arxV4qqqqv0PjjwTte74nM4icTiczmJxETiJxOInGJGMYxjGMYxjGMYxixZxYs4vFXiqvFXm8 Veaq8Xm8VfwtVVXiqqqqr0/jjwTte7ROZzEROYicRE4nERE4icSJEnGMYxjGMYxjGMZxixZxV4qq vF4qrxVXiqqrzeavFX8JVVV4qqqqq9P448E7Xu+IiJzOYnE4iJxEREROIiIk6GJGMYxjGMYxixjO KsXirzV4qrxVXi8VebzVVeb+Gq8VVVVVVXp/HHgna93zOYicxOInERERERERERE5kScYxjGMYxjG cWLOaqqqqqqqqqrxV4q81V5vQq/havFVVVVVV6fxx4J2vd9CcRETicRE4iIiIiJxEREREScYxjGM YxjGMZxYqqqqqqqqqqqqqqvF4qqvTq/havFVVVVVV6fxx4J2vdxOZzERE4iIiIiIiIiIiIiIk4kS JGMYxjGMYsWc2Kqqqqqqqqqqqqq8VVVebzVVfwtXiqqqqqq9P448E7Xu4nM5iIicRERERERERERE REREiRIxjGMYxjFixVVVVVVVVVVVXiqqrxVVV5vNXir+Eq8VVVVVVXp/HHgnad3E5nM5icRERERE RERERERERESJxIxjGMWcVViqqqqqqqqqqqqqqrxV5vN5q8Vfwt4q8VVVVXp/HHgnad2iInM5icRO IiIiIiIiIiIiIiIiRIxjGMWLFVVVVVVVVVVVVVVXirxV5vNVV5q/hKvFVVVVVV6fxx4J2ndoiJxE 5icROIiIiIiIiIiIiIiIiREjGMWLFiqqqqqqqqqqqqqqrxV4q81eKqr0L+EvNVVVVVV6fxx4J2nd oiIiJzE4icREREREREREREREREREjFiqqqqqqqqqqqqqqqqrxV4q81VVVXoX8JeavFVVVV6fxx4J 2ndoiIiJzE4iIiIiIiIiIiIiIiIiIiIxVVVVVVVVVVVVVVVVVVVXirxVVV4q81fwlXirxVVVVen8 ceCdp3aIiIicxOIiIiIiIiIiIiIiIiIiIiVq1VVVVVVVVVVVVVVVVVVV4q8VVVVVeav4SrxV4qqq qvT+OPBO07tEROInMTiJxEREREREREREREREREStWrVVVVVVVVVVVVVVVVV4q8Veaqqqr0L+EvNV VVVVV6fxx4J2ndoiJzOYnETiIiIiIiIiIiIiIiIlRK1rVqrVVVVVVVVVVVVVVVXirxV5q8VVXmr+ EvNVVVVVV6fxx4J2ndxOInM5icROIiIiIiIiIiIiIiIiVK1rWtWrVVVVVVVVVVVVVVVXirxV5vNX i81fwlXiqqqqqq9P448E7Tu4nM5nMTiIiIiIiIiIiIiIiIiVLxrWta1rVq1VVVVVVVVVVVVVVVXi rzebzV4q/hKvFXiqqqq9P448E7Xu4nM5iIicRERERERERERERERESpWta1rWta1aqqqqqqqqqqqq qqqrxVVV5vNVVX8JV4qqqqqqvT+OPBO17tOZzEROYiIiIiJxEREREREvMrWta1rWtatbzaqqqqvF VVVVVeKq8VVVebzeKv4WrxVVVVVVen8ceCdr3fM5iInE5icRERERERERETiJeda1rWta1rW8W81V VVVVXiqqrxV5vFVV5vQv4arxVVVVVVen8ceCdr3fETmInMTiIiJxEREROIiIl6ErWta1rWta1rVq 3m1VVeKqqqrxVVV4q81eLxV5q/havFVVVVVV6fxx4J2vd8TmcziInETiIiInETiIiIlS861rWta1 rWta1vNq1VVVVV4qqqqqrxVXi81eL+JqqqqqqqqvT+OPBO17tE5nMRE6ETiJxE4iInE5iVK3jWta 1rWta1rebWreKvFVVVVVV4q9GqvN5q8VfwlVVXiqqqqr0/jjwTte7nTiJzE4nMTiInERE4nMrWta 1rWta1rWta1a1vN4qqvFVeKvN4q81V6dX8LV4q8VVVVVen8ceCdr3c6E4nM5iczmJxETmcTjW8a1 rWta1rWta1rWta3i3i8W81V4q8VV4qrxebxV5v4arxV4qqqqv0PjjwT/AKz/AOPtV7We1ntd7Ye2 Htl7ae2ntt7ce3Ht17ee3nt9+gfoX6F+ifo36N+kfpX6Z+mfp36h+ofqX6p+qfq36x+sfrX61+uf rn65+vfr369+vfr369+vfrn65+ufrX61+sfrH6t+qfqn6l+ofqH6d+mfpn6V+kfo36N+ifoX6F+g e33t57ee3Xtx7ce23tp7ae2Xth7Ye13tZ7We1XtR7Ue03tR7Ue1HtR7Ue1HtR7Ue1HtR7Ue1HtR7 Ue1HtN7Te03tN7Te0ntJ7Se0ntF7Re0XtB7Qe0Hs97Pezns57NezHst7Leynsl7Iex3sZ7Few3sJ 7Af+fv8Az9/5+/8AP3/n7/z9/wCfvJfQ/wD6n//aAAgBAgIGPwD4YIIIIIIIIIIIIIIIIIIIIIII IIIWLop0U6KdFOinRTop0U6KdFOinRTop0U6KdFPxFPxFPxFfbiK+3EU/EU/EU/EU/EU/EU/EU/E U/EU/EU/EU/EU/EU/EU/EU/EU/8AX+T0PQ2bNmzZs2bNGjRo0aNGjRZZZZZo0aNGjRo0bNmzZs2b Nnoeh6Hoeh6GzZs2bNGjRo0WWWWWX8n/2gAIAQMCBj8A/ZcyKZFMimRTIpkUyKZFMimRTIpkUyKZ FMimRTIpkUyKZFMimRTIpkUyKZFMiq6ZMmTJkyZKKKKKKKKKKK+CiiiiiiiiijJkyZMmTJkwYMmT JkyUUUUUUV+D/9oACAEBAQY/AFGX9kRmThV7jyFXuPIVe48hV7jyFUt5CqW8hVLeQqlvIVS3kKpb yFUt5CqW8hVLeQqlvIVS3kKpbyFUt5CqW8hVLeQqlvIVS3kKpbyFUt5CqW8hVLgKpcBVLeQqlvIV S3kKpbyFUt5CqW8hVLeQqlwFUuAqlwFUuAqlwFUuAqlwFUuAqlwFUuAqlwFUuAqlwFUuAqlwFUuA qlwFUuAqlwFUuAqlwFUt5CqW8hVLeQqlvIVS3kKpbyFUt5CqXAVS4CqW8hVLeQqlvIVS3kKpbyFU t5CqW8hVLeQqlvIVS3kKpbyFUt5CqW8hVLeQqlvIVS3kKpbyFUt5CqW8hVLeQq9x5Cq3HkKvceQq 9x5Cr3HkKvceQq9x5Cr3HkKvceWAq9x5YCr3HlgKvceWAq9x5YCr3HlgKvceWArNx5YCs3HlgKzc eWArNx5YCs3HlgKzceWArNx5YCs3HlgK3ceWArdx5YCt3HlgK3ceWArd15YCu3XlgK7deWArt15Y Cu3XlgK7deWArt15YCu3XlgK7deWArt15YCu3XlgK7deWArt15YCu3XlgK7deWArt15YCu3XlgK7 deWArt15YCu3XlgK7deWArt15YCu3XlgK7deWArt15YCu3XlgK7deWArt15YCu3XlgK7deWArt15 YCu3XlgK7deWArt15YDtL7u8rfUjcSSTV/gjSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7r Ee5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sX lO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucq dLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR 7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU 7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0 sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHu cqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5Tu sR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSx eU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5y p0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6x HucqdLF5TusR7nKnSxeU7rEe5yp0sXlO6xHucqdLF5TusR7nKnSw/wAv6+P5/wCCTESYiTESYiTE SYiTESYiTESoiVESoiVESoiVESoiVESoiVES4iXES4iXES4iXES4iXES4jwiPCI8IjwiPCI8B4Dw HgPAeA8B4jxHiPEeI8R4jwHgPAeA8B4DwHgPCI8IjwiPCIlxEuIlxEuIlxEuIlxEuIlxEqIlREqI lREqIlREqIlREqIlREmIkxEmIkxEmIkxEmIkxEmIkxEiIkREiIkREiIkREiIkREiIkREiIkREiIk REiIkREiIkREiIkREiIkREiIkREiIkREiIkREiIkREiIkREiIkREiI9eI9eI9eI9eI9eI9eI9eI9 eI9eI9aI9aI9b7D1vsPW+w9X7D1fsPV+w9X7D1fsPV+w9X7Dd/8AF/X+v8i/b+r5+Py+P4+fn/TP /9k=" transform="matrix(.106 0 0 -.106 2.825 96.168)"></image></g></g><path fill="#6181B6" d="M64.026 93.694c32.36 0 58.594-13.295 58.594-29.694S96.387 34.306 64.026 34.306 5.433 47.601 5.433 64s26.233 29.694 58.593 29.694z"></path><path fill="#fff" d="M75.896 73.598l2.906-14.958c.656-3.377.11-5.896-1.62-7.486-1.677-1.54-4.523-2.288-8.703-2.288h-5.033l1.44-7.412a.955.955 0 00-.935-1.135h-6.947a.954.954 0 00-.936.771l-3.086 15.881c-.28-1.787-.973-3.323-2.079-4.591-2.038-2.332-5.261-3.515-9.58-3.515H27.856a.951.951 0 00-.935.771L20.674 81.78a.953.953 0 00.935 1.134h7.002a.953.953 0 00.936-.771l1.511-7.775h5.213c2.735 0 5.032-.296 6.827-.881 1.834-.596 3.522-1.607 5.011-3.001a15.364 15.364 0 002.96-3.676l-1.248 6.424a.95.95 0 00.935 1.134h6.947a.954.954 0 00.936-.771l3.429-17.645h4.767c2.031 0 2.626.404 2.787.578.147.159.452.718.11 2.48l-2.764 14.223a.95.95 0 00.935 1.134h7.058a.951.951 0 00.935-.769zm-32.208-12.36c-.437 2.242-1.259 3.842-2.444 4.755-1.205.927-3.132 1.397-5.727 1.397h-3.104l2.244-11.548h4.012c2.948 0 4.136.63 4.612 1.158.662.734.802 2.199.407 4.238zm61.916-8.858c-2.038-2.332-5.261-3.515-9.581-3.515H82.559a.952.952 0 00-.936.771L75.375 81.78a.953.953 0 00.935 1.134h7.003a.953.953 0 00.935-.771l1.512-7.775h5.212c2.735 0 5.033-.296 6.827-.881 1.835-.596 3.522-1.607 5.011-3.001 1.241-1.141 2.264-2.421 3.037-3.806a15.404 15.404 0 001.65-4.588c.797-4.094.16-7.363-1.893-9.712zm-7.262 8.858c-.437 2.242-1.259 3.842-2.444 4.755-1.204.927-3.131 1.397-5.727 1.397h-3.104l2.245-11.548h4.012c2.948 0 4.136.63 4.612 1.158.662.734.801 2.199.406 4.238z"></path><path fill="#000004" d="M38.67 54.89c2.66 0 4.434.491 5.32 1.474.885.982 1.097 2.668.633 5.057-.484 2.488-1.416 4.264-2.798 5.328-1.382 1.063-3.485 1.595-6.308 1.595h-4.26l2.614-13.453h4.799v-.001zM21.609 81.962h7.002l1.661-8.546h5.998c2.646 0 4.823-.277 6.532-.834 1.709-.556 3.263-1.488 4.661-2.797a14.369 14.369 0 002.85-3.569c.727-1.3 1.242-2.734 1.547-4.305.741-3.811.182-6.778-1.676-8.904s-4.812-3.189-8.862-3.189H27.856l-6.247 32.144zm35.394-40.691h6.947l-1.661 8.546h6.189c3.894 0 6.58.68 8.059 2.037 1.479 1.359 1.921 3.561 1.33 6.603l-2.906 14.959h-7.058l2.763-14.223c.314-1.618.199-2.722-.347-3.311-.546-.587-1.708-.882-3.485-.882h-5.553l-3.578 18.416h-6.947l6.247-32.145zM93.324 54.89c2.66 0 4.434.491 5.319 1.474.887.982 1.097 2.668.634 5.057-.484 2.488-1.417 4.264-2.799 5.328-1.382 1.063-3.484 1.595-6.308 1.595h-4.259l2.614-13.453h4.799v-.001zm-17.06 27.072h7.002l1.661-8.546h5.997c2.646 0 4.823-.277 6.532-.834 1.71-.556 3.264-1.488 4.661-2.797a14.35 14.35 0 002.851-3.569c.726-1.3 1.242-2.734 1.547-4.305.74-3.811.182-6.778-1.676-8.904s-4.812-3.189-8.863-3.189H82.511l-6.247 32.144z"></path>'
    SCALA = '<path d="M25 110.437V94.874h5.616c3.113 0 8.052-.203 11.03-.474l5.278-.406-7.038-1.894c-3.924-1.015-8.864-2.504-10.961-3.316L25 87.364V55.627h6.293c3.383 0 8.323-.203 10.894-.473l4.737-.406-8.323-2.233C34 51.366 29.128 49.809 27.639 49.2L25 47.982v-30.72l2.098-.473c1.15-.203 3.992-.406 6.293-.406 11.367 0 38.366-3.722 51.628-7.105 9.27-2.436 15.698-4.872 17.931-6.902 1.15-1.015 1.218-.406 1.218 14.548v15.63l-1.624 1.219-1.624 1.285 3.248 2.842v33.9l-1.624 1.218-1.624 1.286 3.248 2.842v33.9l-1.76 1.353c-1.894 1.489-9.202 3.993-17.524 6.09C71.892 121.737 40.157 126 29.33 126H25z" fill="#390d09"></path><g fill="#de3423"><path d="M25 110.572V95.077l11.842-.474c12.315-.473 21.45-1.488 34.847-3.789 15.225-2.639 30.246-7.375 31.803-10.082.406-.677.676 4.534.676 14.616v15.698l-1.76 1.353c-1.894 1.489-9.202 3.993-17.524 6.09C71.892 121.737 40.157 126 29.33 126H25zM25 71.327V55.83l11.842-.406c13.127-.541 23.344-1.691 36.877-4.195 15.157-2.842 28.96-7.443 29.976-9.947.203-.473.406 6.09.406 14.616.067 13.533-.068 15.698-1.083 16.78-2.368 2.64-20.638 7.376-39.449 10.286-11.435 1.76-30.381 3.79-35.66 3.79H25zM25 32.352V17.195l2.098-.406c1.15-.203 3.992-.406 6.293-.406 11.367 0 38.366-3.722 51.628-7.105 9.27-2.436 15.698-4.872 17.931-6.902 1.15-1.015 1.218-.406 1.218 14.48 0 14.548-.067 15.63-1.285 16.714-1.827 1.691-14.345 5.548-24.09 7.51-15.765 3.113-41.951 6.429-50.883 6.429H25z"></path></g>'
    UNKNOWN = ''

def get_code_language_from_image(svg_code):
    match svg_code:
        case SVG_CODE.C:
            return 'C'
        case SVG_CODE.C_PLUS:
            return 'C++'
        case SVG_CODE.C_SHARP:
            return 'C#'
        case SVG_CODE.JAVA:
            return 'java'
        case SVG_CODE.PYTHON:
            return 'python'
        case SVG_CODE.GO:
            return 'go'
        case SVG_CODE.RUST:
            return 'rust'
        case SVG_CODE.JAVASCRIPT:
            return 'javascript'
        case SVG_CODE.TYPESCRIPT:
            return 'typescript'
        case SVG_CODE.KOTLIN:
            return 'kotlin'
        case SVG_CODE.RUBY:
            return 'ruby'
        case SVG_CODE.SWIFT:
            return 'swift'
        case SVG_CODE.DART:
            return 'dart'
        case SVG_CODE.PHP:
            return 'php'
        case SVG_CODE.SCALA:
            return 'scala'
        case SVG_CODE.UNKNOWN:
            return 'unknown'
        case default:
            print(svg_code)
            raise Exception("No identified Source Language")

In [9]:
def get_code_from_modal(code_element, driver):
    
    try :
    
        driver.execute_script("arguments[0].scrollIntoView();", code_element)

        code_element.click() 

        code_modal = WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "pre")))
        html_code_tag_text = code_modal.get_attribute('innerHTML')

        soup = BeautifulSoup(html_code_tag_text, "html5lib")
        code_text = soup.code.get_text()

        driver.find_element(By.CLASS_NAME, 'modal').send_keys(Keys.ESCAPE)

        return code_text
    
    except :
        
        driver.find_element(By.CLASS_NAME, 'modal').send_keys(Keys.ESCAPE)
        return get_code_from_modal(code_element, driver)

In [10]:
NUM_OF_BIWEEKlY_CONTESTS = 127
NUM_OF_WEEKLY_CONTESTS = 391

def get_leetcode_data():
    
    for num_of_contest in range(NUM_OF_WEEKLY_CONTESTS - 24, NUM_OF_WEEKLY_CONTESTS) :
        for page in range(1,50):
            get_contest_data(num_of_contest, True, page, driver)
        
    for num_of_contest in range(NUM_OF_BIWEEKlY_CONTESTS - 12, NUM_OF_BIWEEKlY_CONTESTS) :
        for page in range(1,50):
            get_contest_data(num_of_contest, False, page, driver)

In [11]:
leetcode_code_df = pd.DataFrame(columns = ['username', 'country','contest_url','num_of_contest', 'finish_time',
                                           'is_weekly','rank', 'score',
                                           'question_1_language', 'question_1_code', 'question_1_finish_time',
                                           'question_2_language', 'question_2_code', 'question_2_finish_time',
                                           'question_3_language', 'question_3_code', 'question_3_finish_time',
                                           'question_4_language', 'question_4_code', 'question_4_finish_time']) 

In [22]:
driver = initialize_driver()

get_leetcode_data()

driver.quit()

Finished scraping  macyapp in contest  385
Finished scraping  Sandeep_17_10_2002 in contest  385
Finished scraping  sourabhmiglani57 in contest  385
Finished scraping  rokanor in contest  385
Finished scraping  vishwas_7 in contest  385
Finished scraping  jendolkk in contest  385
Finished scraping  lijiaqian in contest  385
Finished scraping  shivanshudixit16 in contest  385
Finished scraping  sakuyaqwq in contest  385
Finished scraping  sky_walker-x in contest  385
Finished scraping  Aviral__Gupta in contest  385
Finished scraping  Dengxj in contest  385
Finished scraping  why3881 in contest  385
Finished scraping  jiah in contest  385
Finished scraping  a-mao-14 in contest  385
Finished scraping  user3820h in contest  385
Finished scraping  Keshu1729 in contest  385
Finished scraping  jainshreyansh163 in contest  385
Finished scraping  Bitwise_Operator in contest  385
Finished scraping  mjgupta9135 in contest  385
Finished scraping  jon-snow23 in contest  385
Finished scraping  fourn

Finished scraping  hank55663 in contest  386
Finished scraping  lichen-7i in contest  386
Finished scraping  fplk0 in contest  386
Finished scraping  fmota in contest  386
Finished scraping  klein4groups in contest  386
Finished scraping  tanakat01 in contest  386
Finished scraping  wjli in contest  386
Finished scraping  ting-ting-28 in contest  386
Finished scraping  kuromilover in contest  386
Finished scraping  Rishab01 in contest  386
Finished scraping  AntonRaichuk in contest  386
Finished scraping  awice in contest  386
Finished scraping  OTTFF in contest  386
Finished scraping  LayCurse in contest  386
Finished scraping  zrts in contest  386
Finished scraping  wery0 in contest  386
Finished scraping  htedsv-i in contest  386
Finished scraping  hxu10 in contest  386
Finished scraping  zokumyoin in contest  386
Finished scraping  Chengqian_Li in contest  386
Finished scraping  xinyster in contest  386
Finished scraping  kweezy in contest  386
Finished scraping  deadRabbit in cont

Finished scraping  Xiao__Jun in contest  386
Finished scraping  cwcu in contest  386
Finished scraping  thedude7181 in contest  386
Finished scraping  9caiji in contest  386
Finished scraping  striving_4_ac in contest  386
Finished scraping  sun-man-man in contest  386
Finished scraping  martin0327 in contest  386
Finished scraping  mufti in contest  386
Finished scraping  you-lin-mu-ji in contest  386
Finished scraping  _dipu in contest  386
Finished scraping  saimayank2001 in contest  386
Finished scraping  zenolus in contest  386
Finished scraping  yusufazam in contest  386
Finished scraping  smilences in contest  386
Finished scraping  ryanwong0127 in contest  386
Finished scraping  yuan-zhi-b in contest  386
Finished scraping  viraj_vrj in contest  386
Finished scraping  _whbb in contest  386
Finished scraping  xuan-yuan-long-er in contest  386
Finished scraping  Rohit_Meena in contest  386
Finished scraping  vvizardly-tu in contest  386
Finished scraping  curdking in contest  386

Finished scraping  dnuj4097 in contest  386
Finished scraping  leat14536 in contest  386
Finished scraping  zhou-jing-feng in contest  386
Finished scraping  Arnav_63638 in contest  386
Finished scraping  melonedo in contest  386
Finished scraping  addusirmac in contest  386
Finished scraping  Sanath_Kulla in contest  386
Finished scraping  zhutianqi in contest  386
Finished scraping  juihsiuhsu in contest  386
Finished scraping  Demoralizer_ in contest  386
Finished scraping  huangbin5 in contest  386
Finished scraping  aaronyen in contest  386
Finished scraping  l0vekeven in contest  386
Finished scraping  amol_jindal in contest  386
Finished scraping  muyujx in contest  386
Finished scraping  woodyiiiiiii in contest  386
Finished scraping  fatalerror-i in contest  386
Finished scraping  sdcgvhgj in contest  386
Finished scraping  fangxiang666 in contest  386
Finished scraping  mrn3088 in contest  386
Finished scraping  wannacry89 in contest  386
Finished scraping  Ngxomjy5Qu in cont

Finished scraping  plustone in contest  386
Finished scraping  stupefied-swirlesvfu in contest  386
Finished scraping  pawan_asipu in contest  386
Finished scraping  Gaurav_Upreti in contest  386
Finished scraping  efzfish in contest  386
Finished scraping  auracodehandle in contest  386
Finished scraping  hoanghonghieu in contest  386
Finished scraping  yi1994324 in contest  386
Finished scraping  ninghe in contest  386
Finished scraping  FreeYourMind in contest  386
Finished scraping  brinuke in contest  386
Finished scraping  yuchen1005 in contest  386
Finished scraping  shi-shang-man-ying-hua in contest  386
Finished scraping  wange21 in contest  386
Finished scraping  huangshan01 in contest  386
Finished scraping  Rahul_Gupta07 in contest  386
Finished scraping  flashmt in contest  386
Finished scraping  ln19922 in contest  386
Finished scraping  chanderveersingh099 in contest  386
Finished scraping  sky_akash in contest  386
Finished scraping  magiccircle in contest  386
Finished

...

Finished scraping  la-la-la-v25 in contest  126
Finished scraping  octaneal in contest  126
Finished scraping  gradesking in contest  126
Finished scraping  vibrant-6angulyta3 in contest  126
Finished scraping  bai-ci-2m in contest  126
Finished scraping  hxu10 in contest  126
Finished scraping  kavishd29598 in contest  126
Finished scraping  cheng-liang-yu in contest  126
Finished scraping  yoochun in contest  126
Finished scraping  paw-patrol in contest  126
Finished scraping  XLor in contest  126
Finished scraping  cchao in contest  126
Finished scraping  nevergiveup in contest  126
Finished scraping  aaronyen in contest  126
Finished scraping  Aadersh1 in contest  126
Finished scraping  recursing-almeidafxd in contest  126
Finished scraping  dragonman164 in contest  126
Finished scraping  htedsv-i in contest  126
Finished scraping  ARNOLD66 in contest  126
Finished scraping  bakhtabar in contest  126
Finished scraping  wenboz in contest  126
Finished scraping  cy171 in contest  126

Finished scraping  prashant933 in contest  126
Finished scraping  tttyyy49 in contest  126
Finished scraping  agoodloser in contest  126
Finished scraping  sahilanand in contest  126
Finished scraping  silvertint10 in contest  126
Finished scraping  Selmand42 in contest  126
Finished scraping  Code-O-Maniac in contest  126
Finished scraping  qVUuOvFdPa in contest  126
Finished scraping  looking_for_sde_internships in contest  126
Finished scraping  jmy45 in contest  126
Finished scraping  boring-agnesijl4 in contest  126
Finished scraping  lVLGuRyKq6 in contest  126
Finished scraping  ivanilos in contest  126
Finished scraping  dganguli1997 in contest  126
Finished scraping  TfVQQDhSj6 in contest  126
Finished scraping  qwertyish in contest  126
Finished scraping  hanbro0112 in contest  126
Finished scraping  tai-zhou in contest  126
Finished scraping  sarthak97 in contest  126
Finished scraping  akshay1314 in contest  126
Finished scraping  mrchen116 in contest  126
Finished scraping 

Finished scraping  gregory-2 in contest  126
Finished scraping  smilences in contest  126
Finished scraping  bhardwajb347 in contest  126
Finished scraping  joy32812 in contest  126
Finished scraping  bluez-2 in contest  126
Finished scraping  ham--7 in contest  126
Finished scraping  devashish59 in contest  126
Finished scraping  liuliangcan in contest  126
Finished scraping  amoghprabhu04 in contest  126
Finished scraping  naughty-hodgkinqnn in contest  126
Finished scraping  leek2008 in contest  126
Finished scraping  hkbindal in contest  126
Finished scraping  sunny7712 in contest  126
Finished scraping  sandeepk97 in contest  126
Finished scraping  wei-ji-1 in contest  126
Finished scraping  zi-meng-qin-xiang in contest  126
Finished scraping  shai-zi-8 in contest  126
Finished scraping  kasinathansj in contest  126
Finished scraping  sshivendra764 in contest  126
Finished scraping  Dharun_prakash in contest  126
Finished scraping  Ronak9910 in contest  126
Finished scraping  audi

Finished scraping  cl3424 in contest  126
Finished scraping  yu-shi-b in contest  126
Finished scraping  123_wby in contest  126
Finished scraping  _tim in contest  126
Finished scraping  Ani_S in contest  126
Finished scraping  va14 in contest  126
Finished scraping  nickgeo25 in contest  126
Finished scraping  nikhil_dixit_abv_iiitm in contest  126
Finished scraping  qiu-tian-21bian-17 in contest  126
Finished scraping  rustplayer in contest  126
Finished scraping  JSBL in contest  126
Finished scraping  TejaMeraNaam in contest  126
Finished scraping  4pples in contest  126
Finished scraping  whynot4 in contest  126
Finished scraping  DESGMDkfEQ in contest  126
Finished scraping  SashaKuzin in contest  126
Finished scraping  vivekvar_19 in contest  126
Finished scraping  Yash271120 in contest  126
Finished scraping  chinu7445 in contest  126
Finished scraping  Srajan_Kiyotaka in contest  126
Finished scraping  6ifted-lovelaceqjc in contest  126
Finished scraping  anshul7sh in contest

Finished scraping  nobleknight in contest  126
Finished scraping  stemdarrenyang in contest  126
Finished scraping  srinathsa in contest  126
Finished scraping  funny-clarke2fk in contest  126
Finished scraping  umanghalok in contest  126
Finished scraping  n2n_ in contest  126
Finished scraping  blue_diam0nd01 in contest  126
Finished scraping  Visanth1815 in contest  126
Finished scraping  tangorishi in contest  126
Finished scraping  bayartsogt in contest  126
Finished scraping  awesome-bug in contest  126
Finished scraping  AlphaHokage in contest  126
Finished scraping  lalitms00 in contest  126
Finished scraping  dharshinipg in contest  126
Finished scraping  xzx_123 in contest  126
Finished scraping  NSr_wang in contest  126
Finished scraping  yash_shenoy in contest  126
Finished scraping  saravananbs in contest  126
Finished scraping  vishal_2304 in contest  126
Finished scraping  HARI_PRASAD_2003 in contest  126
Finished scraping  MukulDev007 in contest  126
Finished scraping  

In [23]:
leetcode_code_df.head()

,username,country,contest_url,num_of_contest,finish_time,is_weekly,rank,score,question_1_language,question_1_code,question_1_finish_time,question_2_language,question_2_code,question_2_finish_time,question_3_language,question_3_code,question_3_finish_time,question_4_language,question_4_code,question_4_finish_time
0,fmota,Brazil,https://leetcode.com/contest/weekly-contest-36...,367,00:12:36,True,2,17,C++,class Solution {\npublic:\n vector<int> fin...,00:01:58,C++,class Solution {\npublic:\n string shortest...,00:04:12,C++,class Solution {\npublic:\n vector<int> fin...,00:06:42,C++,class Solution {\npublic:\n vector<vector<i...,00:12:36
1,nicholask_17,Hong Kong,https://leetcode.com/contest/weekly-contest-36...,367,00:13:02,True,3,17,C++,class Solution {\npublic:\n vector<int> fin...,00:05:26,C++,class Solution {\npublic:\n string shortest...,00:07:49,C++,class Solution {\npublic:\n vector<int> fin...,00:05:19,C++,class Solution {\npublic:\n vector<vector<i...,00:13:02
2,skywalkert,China,https://leetcode.com/contest/weekly-contest-36...,367,00:13:24,True,4,17,C++,class Solution {\npublic:\n vector<int> fin...,00:04:36,C++,class Solution {\npublic:\n string shortest...,00:13:24,C++,class Solution {\npublic:\n vector<int> fin...,00:04:25,C++,class Solution {\npublic:\n vector<vector<i...,00:09:25
3,hank55663,Taiwan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:31,True,7,17,C++,class Solution {\npublic:\n vector<int> fin...,00:04:48,C++,class Solution {\npublic:\n string shortest...,00:06:42,C++,class Solution {\npublic:\n vector<int> fin...,00:04:36,C++,class Solution {\npublic:\n const int mod=1...,00:09:31
4,DimmyT,Kazakhstan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:44,True,8,17,C++,class Solution {\npublic:\n vector<int> fin...,00:02:06,C++,class Solution {\npublic:\n string shortest...,00:06:26,C++,class Solution {\npublic:\n vector<int> fin...,00:09:55,C++,class Solution {\npublic:\n vector<vector<i...,00:14:44


In [29]:
leetcode_code_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29732 entries, 0 to 29740
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   username                29732 non-null  object
 1   country                 29732 non-null  object
 2   contest_url             29732 non-null  object
 3   num_of_contest          29732 non-null  int64 
 4   finish_time             29732 non-null  object
 5   is_weekly               29732 non-null  bool  
 6   rank                    29732 non-null  object
 7   score                   29732 non-null  object
 8   question_1_language     29708 non-null  object
 9   question_1_code         18333 non-null  object
 10  question_1_finish_time  29708 non-null  object
 11  question_2_language     29551 non-null  object
 12  question_2_code         18231 non-null  object
 13  question_2_finish_time  29551 non-null  object
 14  question_3_language     27475 non-null  object
 15  questio

In [25]:
def get_user_global_rank(username):
    time.sleep(0.5)
    
    try:
        
        driver.get(f'https://leetcode.com/{username}/')
        
        if(len(driver.find_elements("xpath","//img[@alt='404 not found']")) > 0) :
            print(f'{username} profile not exist in the site')
            return None
        
        rank = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "ttext-label-1"))).text.replace(',', '')
        if rank.startswith('~'):
            rank = rank[1:]
        print(f'{username} rank is {rank}')
        return int(rank)
    except:
        time.sleep(5)
        return get_user_global_rank(username)
    

In [30]:
driver = initialize_driver()
leetcode_code_df['user_global_rank'] = leetcode_code_df['username'].apply(get_user_global_rank)
driver.quit()

fmota rank is 486427
nicholask_17 rank is 27684
skywalkert rank is 16
hank55663 rank is 6234
DimmyT rank is 702585
waakaaka rank is 555875
LayCurse rank is 8769
flashmt rank is 228225
natsugiri rank is 9865
tmoux rank is 194518
DylanSmith rank is 578
Aging rank is 19520
plevande rank is 3613
i_love_xiaoshagua_cpp rank is 3279
finite_automaton_enthusiast rank is 26552
zanj0 rank is 90
Rohit2593 rank is 10904
non_deterministic rank is 78586
sreeshmaheshwar rank is 199465
arsh_agarwal rank is 420312
nikatamliani1 rank is 764617
zrts rank is 282849
borat89 rank is 1747987
hkmaxi rank is 1095
BossBobster rank is 12007
biharicoder rank is 953
_priyanshu_101_ rank is 77043
hxu10 rank is 30
SaveVMK rank is 2545
OTTFF rank is 154
gracesharma47 rank is 419831
DansGame rank is 1032
twitch_tv_qiqi_impact rank is 2664
yatin_kwatra rank is 6746
cai_lw rank is 1425
R3mix rank is 1240
u77 rank is 144
donbasta rank is 2499
wismer rank is 675356
inversionpeter rank is 1053
subhra_29 rank is 59324
Tinky1

8symbols rank is 1923
roytanmay rank is 11228
mohit__09 rank is 6182
satyamaman865 rank is 9779
88ion rank is 36038
bluesky999 rank is 137
zihao_intuition rank is 53607
parasNAGPAL99 rank is 3511
shaheen_bd rank is 1914
cjyang1230 rank is 703
tourist_fan_fan_fan rank is 9555
ashish_2298744 rank is 4653
bharatkhanna29 rank is 1110433
PrakshalJainn rank is 61434
Finesse rank is 20858
jeckafedorov rank is 5793
forsakencode rank is 6031
shinever rank is 6196
hahahiehie rank is 381
Lelouch_17 rank is 477565
Akash_Ashok_11 rank is 6125
yumichael rank is 11623
vermachakshu rank is 21109
garglakshay048 rank is 54740
CodeFella rank is 94533
TheRaven rank is 79410
leftshifted rank is 5877
dhruvan111 rank is 282266
Wooohs rank is 9948
shivambhagat02 rank is 105933
rohitt28 rank is 370363
fizhim rank is 12387
ashif62224 rank is 18949
endless_mercury rank is 2003
omtiwari rank is 89525
Harsh_Aakoliya rank is 142353
SathishBatsy rank is 23254
Ankitsingh_20 rank is 345486
mugicha101 rank is 8774
Om_A

Ar_2000 rank is 7513
Dharun_prakash rank is 65011
AmanJha04527 rank is 80347
SP__21 rank is 8049
pseudonmos rank is 40630
webdevesh819 rank is 123450
yash_karakoti rank is 379099
geekyuttu rank is 202816
ayushmanbajpayee10 rank is 6004
qize rank is 4720
bhavya_kawatra13 rank is 35266
mkawa222 rank is 1150
Sahajgup rank is 259366
rishabg30 rank is 13736
haeundae96 rank is 10994
ayushsaxena823 rank is 63490
shiv99265 rank is 7418
ye15 rank is 45
Ranxin2024 rank is 31011
BossBobster rank is 12007
AntonRaichuk rank is 110418
nicholask_17 rank is 27684
uwi rank is 1219
wery0 rank is 23458
natsugiri rank is 9865
Cookie_Cream rank is 238539
LayCurse rank is 8769
donbasta rank is 2499
SaveVMK rank is 2545
roma95 rank is 40660
OTTFF rank is 154
mikeqiyh rank is 674336
zephyrr rank is 10350
XAXAEBATb rank is 255359
raincoat911 rank is 145
cchao rank is 11388
y0105w49 rank is 627017
kefaa rank is 42620
waakaaka rank is 555875
DimmyT rank is 702585
borat89 rank is 1747987
szp14 rank is 250
megaspa

jitendrajitu0563 rank is 167299
sakthivel_mani rank is 1268
shixun404 rank is 318555
k_shark_ocean rank is 14472
Tinky1224 rank is 380
hnxxzhang rank is 157326
huanli01008 rank is 40810
dorfi rank is 48014
sudhanshu_090 rank is 65900
panu_10 rank is 184799
akhildhatterwal rank is 264960
cp_boy rank is 349390
ac130h rank is 1587
cinoss rank is 110525
Golu-bhai rank is 1219534
Rahul9588 rank is 4495
log1 rank is 12987
danzhi rank is 82
athulzkrish rank is 4757
ashwanthkannan rank is 108937
Amangupta2003 rank is 75413
kaustubhreet rank is 19315
ryanwong0127 rank is 1966
Almighty333 rank is 55076
HTG-18 rank is 122904
mishrakripanshu303 rank is 20044
shinever rank is 6196
chirag0123 rank is 43152
randytanpty rank is 148
me356500 rank is 15594
ajeetVerma12 rank is 99830
ambuj21 rank is 66782
sourav_kd rank is 2441
xzaviourr rank is 412292
soumya_2k01 rank is 97063
Sanjay_Seemakurti rank is 983290
SalvadorDali rank is 328
biharicoder rank is 953
tandonjiii rank is 14647
deveshanurag rank is 

LGM70 rank is 181122
Himanshu_Kaithal rank is 12771
vaidehij92 rank is 231749
mayankaryan rank is 42062
zhouwei0422 rank is 932
pawanverma25 rank is 5103
puriabhijit000 rank is 4040
rahul0409 rank is 400349
slothalan rank is 1731
Wooohs rank is 9948
vivbhave rank is 10924
kingsman007 rank is 65285
shannonl rank is 4674
eLeet_chirag rank is 54472
tagorganesh464 rank is 189348
KidPooh rank is 4541
babluisnotbodmash rank is 8875
shikshrey rank is 353049
shrey0811 rank is 10959
emanuel_sygal rank is 279630
vimaldhiman rank is 71390
pankaj_777 rank is 14230
Biswajit_Kaushik rank is 137431
ericet1234 rank is 1852
endless_mercury rank is 2003
manish26eleven rank is 591226
rishcusion131072 rank is 40107
Sana1278 rank is 805968
maxinimus rank is 106483
Atul_Raj rank is 15242
singhalPratham rank is 28397
Kalki4568 rank is 279422
chenjun15 rank is 32445
TlTAN rank is 13510
Anish-Karthik rank is 38259
adarsh_20_03 rank is 64150
Nilesh_1206 rank is 2990
kadamomkar rank is 7813
AaryanSaraswat rank i

...

Vansh_Kapila rank is 32173
vanshsuneja_ rank is 51649
AtulKeshari rank is 45846
Arjun_Yadav02 rank is 360477
saidinesh01 rank is 1884825
utkarsh_s_sood rank is 88626
kaarti_22 rank is 309208
dennis753951 rank is 732
fr08 rank is 382366
xil899 rank is 681
yabanci rank is 192994
AdityaDargan rank is 375612
BitWizz rank is 133391
log1 rank is 12732
yeyunfeng-hub profile not exist in the site
yatin_kwatra rank is 6761
Dhongee rank is 95029
helltractor profile not exist in the site
adiboy77 rank is 61532
Maanas23 rank is 14884
s_mtLC rank is 539131
sgc109 rank is 188057
21R01A0426 rank is 1280285
awesson rank is 153842
Astatine08 rank is 601239
erenaltunoglu rank is 95974
JainWinn rank is 13908
debugger612 rank is 75480
RRoy30 rank is 228656
Celestial_Coder rank is 8902
karanrana3095 rank is 129532
bugwei rank is 69392
robot810975 profile not exist in the site
saptaswadas69 rank is 6934
agitated-ardinghellixfi profile not exist in the site
nvladgw rank is 141842
relaxed-yalowzt2 profile not

mukeshbarpete4 rank is 18187
dsjong rank is 298586
hellraiser666 rank is 3403
aman_2005 rank is 4188
man-ray rank is 5000000
kartikyadav_12 rank is 350117
manoj183 rank is 7094
Gaurav200 rank is 769578
sykobeli rank is 31922
_harshit_soni_ rank is 1180781
p4nda rank is 139947
infinity0103 rank is 72572
rushikeshbiradar rank is 498517
lxc-km profile not exist in the site
skqliiiao profile not exist in the site
kartikeyy rank is 52754
amitSha rank is 19809
priyanshur754 rank is 539950
manas_47 rank is 450029
harshit7650 rank is 332341
_MdSourav_ rank is 119233
heysaiyad rank is 389489
zoehuhu profile not exist in the site
tanishq02 rank is 2771427
mainakhalder rank is 138323
ggstddu profile not exist in the site
intelligent-teslas5c profile not exist in the site
kind-belllxh profile not exist in the site
gabrudj rank is 113723
Chengqian_Li rank is 4609
Katyaldeepak rank is 75236
agrim07 rank is 378053
rudro25 rank is 26207
gxylengli rank is 5000000
songhayoung0_0 rank is 829152
biharicod

none-31 profile not exist in the site
00zi-jian profile not exist in the site
summerdaway rank is 1923221
omipus profile not exist in the site
Ruiko rank is 5000000
hughstudy-n profile not exist in the site
Skywarrior2000 rank is 13898
thedude7181 rank is 2530
bofeng07 rank is 35
jwseph rank is 8766
subcrip profile not exist in the site
han3000 rank is 189919
zhzho profile not exist in the site
fuwutu rank is 346
youngk-2 profile not exist in the site
gaut_2003 rank is 10383
zronghui profile not exist in the site
la-la-la-v25 profile not exist in the site
octaneal rank is 22674
gradesking profile not exist in the site
vibrant-6angulyta3 profile not exist in the site
bai-ci-2m profile not exist in the site
hxu10 rank is 33
kavishd29598 rank is 5763
cheng-liang-yu profile not exist in the site
yoochun profile not exist in the site
paw-patrol profile not exist in the site
XLor rank is 5000000
cchao rank is 11411
nevergiveup rank is 149
aaronyen rank is 1960
Aadersh1 rank is 95849
recursin

cpp20 rank is 5000000
Rad0miR rank is 3413
saberjiang-b profile not exist in the site
wangwenwwx rank is 2257109
Raja_n rank is 5448
zhao-xi-20 profile not exist in the site
Abhi_Srivastava rank is 174646
slizer rank is 91189
flamboyant-i2aman7jl profile not exist in the site
smomal profile not exist in the site
Vansh_Kapila rank is 32173
dush1729 rank is 3161
raghava118 rank is 14374
981377660LMT rank is 408196
ou-hai-ziJHu23dNZ profile not exist in the site
chanpreetug20 rank is 800866
andy-d7 profile not exist in the site
arjit_07 rank is 84636
igor_kz rank is 32826
FreeYourMind rank is 196119
harsh_reality_ rank is 8218
sad-hodgkin23h profile not exist in the site
navyakatiyar rank is 77230
vinayak82 rank is 312607
taoz-kc profile not exist in the site
2000bhawesh rank is 630223
kanoon profile not exist in the site
nguyenducloc rank is 7341
chuchuching rank is 11112
Ronak_Ramuka rank is 20311
Aryam_singh rank is 67261
Astatine08 rank is 601239
tdkkdt rank is 6253
just_hands13 rank 

cl3424 rank is 1796
yu-shi-b profile not exist in the site
123_wby profile not exist in the site
_tim rank is 5000000
Ani_S rank is 2664
va14 rank is 5677
nickgeo25 rank is 389764
nikhil_dixit_abv_iiitm rank is 57138
qiu-tian-21bian-17 profile not exist in the site
rustplayer profile not exist in the site
JSBL rank is 2233549
TejaMeraNaam rank is 170243
4pples profile not exist in the site
whynot4 rank is 556538
DESGMDkfEQ profile not exist in the site
SashaKuzin rank is 122595
vivekvar_19 rank is 4386
Yash271120 rank is 176934
chinu7445 rank is 330487
Srajan_Kiyotaka rank is 17125
6ifted-lovelaceqjc profile not exist in the site
anshul7sh rank is 2088
Assassin-Killer profile not exist in the site
Aman_G011 rank is 34332
chi-xian-k profile not exist in the site
singhal03 rank is 11795
Shivesh__Gupta__ rank is 44079
shivansh_3950 rank is 6664
windyyxz rank is 473551
josephstarkrgb rank is 5536
fangxiang666 profile not exist in the site
argha_kole rank is 1146
jakubd rank is 56522
nemm r

In [31]:
leetcode_code_df.head()

,username,country,contest_url,num_of_contest,finish_time,is_weekly,rank,score,question_1_language,question_1_code,...,question_2_language,question_2_code,question_2_finish_time,question_3_language,question_3_code,question_3_finish_time,question_4_language,question_4_code,question_4_finish_time,user_global_rank
0,fmota,Brazil,https://leetcode.com/contest/weekly-contest-36...,367,00:12:36,True,2,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:04:12,C++,class Solution {\npublic:\n vector<int> fin...,00:06:42,C++,class Solution {\npublic:\n vector<vector<i...,00:12:36,486427.0
1,nicholask_17,Hong Kong,https://leetcode.com/contest/weekly-contest-36...,367,00:13:02,True,3,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:07:49,C++,class Solution {\npublic:\n vector<int> fin...,00:05:19,C++,class Solution {\npublic:\n vector<vector<i...,00:13:02,27684.0
2,skywalkert,China,https://leetcode.com/contest/weekly-contest-36...,367,00:13:24,True,4,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:13:24,C++,class Solution {\npublic:\n vector<int> fin...,00:04:25,C++,class Solution {\npublic:\n vector<vector<i...,00:09:25,16.0
3,hank55663,Taiwan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:31,True,7,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:06:42,C++,class Solution {\npublic:\n vector<int> fin...,00:04:36,C++,class Solution {\npublic:\n const int mod=1...,00:09:31,6234.0
4,DimmyT,Kazakhstan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:44,True,8,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:06:26,C++,class Solution {\npublic:\n vector<int> fin...,00:09:55,C++,class Solution {\npublic:\n vector<vector<i...,00:14:44,702585.0


In [32]:
leetcode_code_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29732 entries, 0 to 29740
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   username                29732 non-null  object 
 1   country                 29732 non-null  object 
 2   contest_url             29732 non-null  object 
 3   num_of_contest          29732 non-null  int64  
 4   finish_time             29732 non-null  object 
 5   is_weekly               29732 non-null  bool   
 6   rank                    29732 non-null  object 
 7   score                   29732 non-null  object 
 8   question_1_language     29708 non-null  object 
 9   question_1_code         18333 non-null  object 
 10  question_1_finish_time  29708 non-null  object 
 11  question_2_language     29551 non-null  object 
 12  question_2_code         18231 non-null  object 
 13  question_2_finish_time  29551 non-null  object 
 14  question_3_language     27475 non-null  obj

In [33]:
leetcode_code_df.to_csv('leetcode_code_df.csv')